# The Impact of M&A on Inventor Mobility and Innovation
## Draft notebook for the GitHub package

This notebook is a narrative companion to the construction and analysis pipeline in the repository. It is written to do four jobs at once:

1. explain how the data are built and how the main empirical designs work,
2. present a disciplined headline story,
3. distinguish robust findings from fragile or method-dependent ones,
4. serve as a base for a sequence of public-facing writeups such as LinkedIn posts.

## Current headline story

The cleanest story is **not** that mergers and acquisitions uniformly reduce innovation output.  
The stronger and more defensible claim is narrower:

> **M&A appears to change inventor mobility and the direction of inventive search more clearly than it changes aggregate output, with especially important effects on the acquiror side.**

In particular, the most interesting pattern is that **acquiror-side inventor mobility rises after deals, and exploration becomes more prominent**. By contrast, effects on patents, citations, and related output measures are often more fragile across methods or complicated by pre-trends.

## What this notebook treats as primary versus secondary

### Primary findings
- Acquiror-side post-deal inventor mobility.
- Acquiror-side exploration outcomes.
- Heterogeneity by firm size, especially where negative baseline effects weaken among larger firms.

### Secondary findings
- Aggregate output measures such as patent counts, citations, and quality proxies.
- Arrivals and departures as supporting evidence for organizational reshuffling.
- Target-side effects, unless they are unusually stable across methods.

## Important caution

This draft reflects the current state of the project and the review notes. Some statements below are intentionally cautious because a few results are sensitive to estimator choice, event-study shape, or pre-trend concerns.

## Repository orientation

A clean GitHub version of the project should separate the work into:

- **construction scripts**: build cleaned firm-year and inventor-year / inventor-event panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,
- **notebooks**: one main narrative notebook, plus optional shorter diagnostics notebooks.

This notebook is meant to be the **main interpretive notebook** rather than the place where every heavy computation happens.


## Construction of the dataset: overview and design logic

A major part of the value of this project is the construction itself. The empirical design is only credible if the data pipeline is transparent about how raw patent, accounting, and M&A records are converted into analysis-ready panels.

This section therefore documents the construction in the same spirit as a methods section in an academic paper, but with a stronger emphasis on engineering choices, reproducibility, and economic interpretation. The target reader is a technically literate economist, data scientist, or recruiter who wants to understand not only *what* was estimated, but *how* the underlying research asset was built.

### What the construction is trying to achieve

The pipeline solves four linked problems:

1. **Link inventions to inventors and firms.**  
   Patent records are inventor-centric and assignee-centric, while the firm-level analysis requires a stable public-firm identifier.

2. **Build meaningful innovation measures rather than just patent counts.**  
   The project adds citation-based quality proxies, novelty measures, exploration/exploitation metrics, and technology proximity scores.

3. **Identify economically meaningful inventor moves.**  
   The mobility analysis is not based on one-off assignee changes. It uses a stricter move definition designed to reduce false positives from noisy patent assignment histories.

4. **Create panels aligned to M&A timing.**  
   The end product is not merely a collection of cleaned files. It is a set of firm-year, inventor-year, and inventor-event panels centered on merger timing and suitable for DiD and event-study analysis.

### Why a multi-stage pipeline is necessary

No single source contains the final object needed for the project.

- **PatentsView** provides patent, inventor, assignee, location, CPC, and citation information.
- **External patent-quality data** add KPSS-based measures such as `xi_real` and citation fields.
- **Compustat** provides firm fundamentals.
- **WRDS link tables** connect Compustat firms to CRSP/market identifiers such as `permco`.
- **SDC M&A data** provide acquisition and target events.

The pipeline therefore moves from raw source-specific files to increasingly integrated research objects:

> raw patent and firm data → enriched patent-inventor-firm file → mobility and technology measures → final firm-year and inventor-event analysis panels

### Construction philosophy

Three choices are worth stating up front.

**First, the pipeline is intentionally conservative about identifiers.**  
The core analytical firm identifier is `permco`, because it gives a stable public-firm identifier that can be used across patents, market data, and M&A events.

**Second, intermediate files are cached aggressively.**  
This is partly a speed issue and partly a reproducibility issue. Some steps are expensive enough that a serious empirical workflow benefits from explicit checkpoints.

**Third, the split repository structure preserves the logic of the original monolithic construction script.**  
The runner executes modules in sequence using a shared namespace. That is not the most object-oriented design possible, but it is a sensible choice for a research codebase where preserving validated logic is often more important than abstract elegance.



## Construction pipeline at a glance

The construction code is organized into eight sequential modules plus one runner script.

| File | Main purpose | Main outputs |
|---|---|---|
| `run_construction.py` | Executes all construction modules in order | End-to-end pipeline |
| `01_setup_helpers.py` | Paths, imports, global helpers, data download helper | Shared environment |
| `02_patent_panel_construction.py` | Builds the core patent-inventor-firm dataset and patent-level measures | `pat_inv_firm_df.pkl` and intermediate patent objects |
| `03_exploration_exploitation.py` | Adds exploration/exploitation metrics | enriched patent-inventor-firm file |
| `04_mobility_and_mover_metrics.py` | Detects inventor moves and builds move-related performance objects | mover event and move-performance files |
| `05_technology_similarity.py` | Computes technology proximity and rolling similarity measures | event-based and rolling similarity outputs |
| `06_firm_fundamentals.py` | Builds Compustat-based firm fundamentals and links them to `permco` | linked Compustat panel |
| `07_linking_and_manda.py` | Cleans and links M&A deals and adds pre-deal technology similarity | `manda.pkl` |
| `08_final_panels.py` | Produces the final firm-year, firm-event, inventor-year, and inventor-event panels | final analysis panels |

### Pipeline orchestration

The runner preserves the original notebook-like dependency structure:

```python
EXECUTION_ORDER = [
    "01_setup_helpers.py",
    "02_patent_panel_construction.py",
    "03_exploration_exploitation.py",
    "04_mobility_and_mover_metrics.py",
    "05_technology_similarity.py",
    "06_firm_fundamentals.py",
    "07_linking_and_manda.py",
    "08_final_panels.py",
]
```

and then executes the files in a **shared global namespace**:

```python
shared_globals = runpy.run_path(
    str(script_path),
    init_globals=shared_globals,
    run_name="__main__",
)
```

This is a deliberate research-engineering compromise. It keeps the split files readable and GitHub-friendly while minimizing the risk of introducing logic changes during refactoring.



## Step 0. Environment, paths, and helper infrastructure

The setup module does four jobs:

1. loads the core Python stack,
2. defines the project paths,
3. validates that required local inputs exist,
4. provides a helper to download PatentsView files on demand.

### Main libraries used

This construction is fundamentally a **Python data engineering and empirical research pipeline**. The main tools are:

- **pandas** and **NumPy** for table manipulation and vectorized operations,
- **scikit-learn** for matching support and nearest-neighbor logic,
- **tqdm** for progress monitoring in long-running loops,
- **requests** and **zipfile** for downloading and unpacking PatentsView files,
- **collections.Counter / defaultdict / deque** for efficient rolling technology-history objects.

The setup code also defines a strict path check:

```python
def assert_required_paths_exist():
    required_paths = [
        BASE_PROJECT_PATH,
        RAW_DATA_PATH,
        FINANCIAL_DATA_PATH,
        MANDA_DATA_PATH,
        LINKTABLE_CSV,
    ]
    for path in required_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required path does not exist: {path}")
```

This matters because the project depends on several local raw-data directories. Failing early is better than discovering a missing file after several hours of preprocessing.

### On-demand raw-data loading

A useful design choice is the helper that downloads PatentsView files only if they are not already stored locally:

```python
def download_and_load_patentsview_data(file_name, **kwargs):
    base_url = 'https://s3.amazonaws.com/data.patentsview.org/download'
    local_file_path = os.path.join(RAW_DATA_PATH, file_name)
    if os.path.exists(local_file_path):
        print(f"Loading '{file_name}' from local directory.")
    else:
        r = requests.get(f"{base_url}/{file_name}.zip", timeout=300)
        r.raise_for_status()
        with zipfile.ZipFile(BytesIO(r.content)) as z:
            z.extractall(RAW_DATA_PATH)
    return pd.read_csv(local_file_path, delimiter="\t", ...)
```

Economically, this step is mundane. Methodologically, it is important. It makes the project reproducible from raw public patent files instead of relying on opaque manual extracts.

### Why this matters for the notebook narrative

For an employer-facing research notebook, this section demonstrates that the project is not just a set of regressions. It is a full empirical data asset with explicit input validation, caching, and recoverability.



## Step 1. Build the core patent-inventor-firm dataset

This is the foundational construction step. Everything later in the project depends on producing a clean patent-level file that links:

- a patent,
- its inventor(s),
- the public firm to which it can be assigned,
- the patent's technological and quality characteristics.

### Raw patent inputs

The construction draws the following core tables from PatentsView:

- inventor-patent links,
- assignee-patent links,
- inventor locations,
- CPC technology classifications,
- patent-to-patent citations,
- patent issue dates,
- application filing dates.

The code begins with a cache-first wrapper:

```python
def build_or_load_pat_inv_firm(cache_path=None, rebuild=False):
    if (not rebuild) and os.path.exists(cache_path):
        return pd.read_pickle(cache_path)
```

This is good empirical practice because the patent core is both expensive and stable. Once validated, it should not be rebuilt unless upstream logic changes.

### Cleaning and identifier discipline

Several cleaning choices are substantive rather than cosmetic.

#### 1. Keep business assignees and the primary assignee
The assignee file is filtered to:

```python
assignee_df = assignee_df[
    (assignee_df['assignee_type'] == 2) &
    (assignee_df['assignee_sequence'] == 0)
].copy()
```

This means the construction is intentionally focused on the primary business assignee rather than all possible assignee relationships. That is a defensible choice because the downstream goal is to connect patents to publicly listed firms in a way that is stable enough for event-study analysis.

#### 2. Use filing year rather than issue year for many research objects
The code merges issue dates and application dates and defines:

```python
patent_df['filing_year'] = patent_df['filing_date'].dt.year
```

This is economically sensible because filing dates are closer to the timing of inventive activity than grant dates, which can be delayed by examination and administrative processes.

#### 3. Retain detailed CPC information
The code keeps CPC subclasses and groups, using primary CPC assignments for some tasks and full CPC combinations for novelty construction later. This matters because technology direction is central to the project's mechanism story.

### External enrichments

Two external merges are especially important.

#### KPSS patent-quality data
The construction merges patent-level `xi_real` and citation information from an external replication dataset and links patents to `permco`. This is how the project transitions from patent objects to firm-identified patent objects.

#### State-level noncompete enforceability
The code merges a state-year noncompete enforcement score using inventor state and filing year:

```python
pat_inv_firm_df = pat_inv_firm_df.merge(
    nca_df[['filing_year', 'state_fips', 'std_score']],
    on=['filing_year', 'state_fips'], how='left'
).rename(columns={'std_score': 'nca_enforce_score'})
```

This is a useful example of economically informed enrichment. The project is about inventor mobility, so including a local legal environment relevant to mobility is conceptually well motivated.

### Patent-level quality measures

The construction does not rely on one patent-quality measure. It builds several.

#### Forward citations
First, unconditional forward citations are computed from citation links. Then the code constructs class-year normalized bins based on CPC subclass and filing year:

```python
stats = df.groupby(['filing_year', 'cpc_subclass'])['forward_citations']           .quantile([0.90, 0.99]).unstack()
```

This is good measurement design. Raw citations are noisy across technology classes and cohorts. Ranking patents within technology-year cells makes the quality proxy more comparable.

#### KPSS-based citation bins
The same binning logic is repeated for the KPSS citation field. This redundancy is useful: it reduces dependence on a single citation definition.

### Patent novelty

The novelty measure is based on new combinations of CPC classes, following the logic of recombinatorial innovation. The key idea is simple:

- represent a patent as a set of technology classes,
- enumerate all within-patent class pairs,
- identify whether each pair is new in the historical record,
- summarize the share of new combinations.

A simplified version of the core logic is:

```python
def _canonical_pair(a, b):
    return f'{a}_{b}' if a <= b else f'{b}_{a}'

pairs = g.assign(pair_key=lambda df: df['cpc_tuple'].apply(_make_pairs)).explode('pair_key')
first_date_per_pair = pairs.groupby('pair_key')['issue_date_dt'].min()
```

This is an intellectually strong part of the construction because it goes beyond volume and average quality. It directly measures whether the patent recombines knowledge in a way that appears novel relative to prior art.

### Citation-link measures

The pipeline also builds:

- backward citations,
- self-citations at the firm level.

The self-citation routine is written carefully to be memory-safe by operating in chunks and comparing assignee sets through set intersections. That is a strong engineering choice for large citation graphs.

### Final patent-inventor-firm object

After all merges, the project creates the core research object:

```python
pat_inv_firm_df = pat_inv_df.merge(patent_df, on='patent_id', how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(kpss_df, on=['patent_id'], how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(patent_novelty_df, on='patent_id', how='left')
...
```

It then adds inventor career features such as:

- first filing year,
- first CPC field,
- inventor age measured as years since first filing,
- indicators for whether a patent stays close to the inventor's original field.

### Why this step is done this way

This construction step is designed to create a **single enriched micro-level innovation file** from which multiple downstream panels can be derived without rebuilding patent logic each time. That is both computationally efficient and methodologically cleaner.



## Step 2. Construct exploration and exploitation measures

One of the most interesting aspects of the project is that it tries to distinguish **direction of search** from **quantity of output**.

The key intuition is that mergers may reshape what inventors and firms work on even when the total number of patents does not move much.

### Measure design

For each patent, the code builds a set of CPC subclasses and compares that set to the recent knowledge base of:

- the patent's inventor team,
- the patent's firm or firms.

The knowledge base is defined using a rolling five-year window.

A simplified version of the logic is:

```python
for row in patent_level.itertuples(index=False):
    inv_knowledge = union_of_recent_cpcs_for_inventors
    firm_knowledge = union_of_recent_cpcs_for_firms

    exp_inv = 1 - overlap(current_cpcs, inv_knowledge) / len(current_cpcs)
    exp_firm = 1 - overlap(current_cpcs, firm_knowledge) / len(current_cpcs)
```

### Why this is economically meaningful

A patent is classified as more exploratory when it uses CPC classes that are less represented in the inventor's or firm's recent history. This is an intuitive way to measure movement into less familiar technological space.

This is also one reason the project's headline story is more persuasive for exploration than for aggregate patent counts. Exploration is closer to the organizational mechanism one might expect after an acquisition:

- new teams are combined,
- internal allocation changes,
- inventors may be redeployed,
- firms may search more broadly or in adjacent spaces.

### Why the implementation uses rolling deques

The implementation stores prior technology histories in `deque` objects and purges outdated years as it moves through the patent sequence. That makes the rolling-window computation efficient enough to scale while remaining transparent.



## Step 3. Identify inventor mobility events and build mover metrics

A central contribution of the project is that inventor mobility is not treated as a noisy byproduct. It is directly measured and then connected to post-M&A firm outcomes.

### Strict move identification

The move definition is intentionally conservative. The code first restricts attention to inventors with at least five patents:

```python
min_patents = 5
prolific_inventors = inventor_counts[inventor_counts >= min_patents].index
```

Then it defines a move only when there is a stable firm sequence around the transition:

- two patents before the move at the old firm,
- the first patent at the new firm,
- two subsequent patents at the new firm.

The core rule is:

```python
is_move = (
    (career_df['permco'] != career_df['prev_permco']) &
    (career_df['prev2_permco'] == career_df['prev_permco']) &
    (career_df['next_permco'] == career_df['permco']) &
    (career_df['next2_permco'] == career_df['next_permco'])
)
```

### Why this strict rule is useful

Patent assignment data can be noisy. A single assignee switch may reflect legal assignment timing, joint work, or temporary data noise rather than a real labor-market transition.

The stricter move rule sacrifices some sample size, but it likely improves signal quality, which is exactly the right tradeoff for a project whose main mechanism involves labor reallocation.

### Pre/post mover performance

Once moves are defined, the code builds five-year pre-move and post-move performance windows and aggregates inventor outcomes such as:

- patent counts,
- citations,
- `xi_real`,
- novelty,
- backward and self-citations,
- exploration and exploitation,
- team size.

This produces both:
- a long panel with one row per inventor-move-period, and
- a wide format that makes change calculations straightforward.

### Benchmarking movers relative to peers

The project also compares movers to peers at origin and destination firms. That is a subtle but valuable addition because it helps distinguish:

- whether firms are losing unusually important inventors,
- whether incoming inventors look stronger or weaker than incumbent peers,
- whether post-move performance changes suggest integration frictions or gains.

From a research-design perspective, this is a strong bridge between micro labor allocation and firm-level organizational outcomes.



## Step 4. Measure technology similarity around moves and deals

The project next constructs technology-similarity objects that help interpret whether mobility and M&A connect technologically related or unrelated domains.

### Event-based proximity around inventor moves

The first routine compares technology vectors before and after a move for:

- the inventor,
- the origin firm,
- the destination firm.

Vectors are represented as `Counter` objects over CPC subclasses, and similarity is measured with cosine similarity:

```python
def _calculate_cosine_similarity(vec1, vec2):
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    norm1 = sqrt(sum(v**2 for v in vec1.values()))
    norm2 = sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2)
```

This yields quantities such as:

- inventor pre/post self-similarity,
- inventor similarity to origin firm,
- inventor similarity to destination firm,
- origin-destination firm similarity.

### Rolling annual similarity

The second routine creates a rolling annual similarity measure, comparing a current year's technology vector with the preceding five-year technology history. This is helpful because it turns technology direction into a panel variable rather than a one-time event summary.

### Why these measures matter

These objects are not the headline outcomes of the project, but they greatly improve interpretation. If mobility rises after deals, the next question is whether it reflects:

- redeployment into familiar technologies,
- integration into the acquiring firm's technological base,
- broader exploratory search.

Technology similarity measures help answer that question.



## Step 5. Build firm fundamentals from Compustat and link them to market identifiers

The firm-side empirical analysis needs more than patent outcomes. It needs standard controls and heterogeneity variables describing firm size, profitability, leverage, liquidity, investment, and financial constraints.

### Sample construction in Compustat

The code filters Compustat to a standard industrial sample:

- excludes financials, utilities, and public sector entities,
- keeps industrial, standardized, consolidated, domestic statements,
- removes clearly invalid negative values for core accounting variables.

It then defines the analysis year as:

- same calendar year if the fiscal year ends in June to December,
- previous calendar year otherwise.

That timing rule is sensible because it aligns fiscal records more closely with the year to which they economically belong.

### Feature engineering

The module then constructs a broad set of variables, including:

- size: `log_at`, `log_sale`, `log_mv`,
- profitability: `roa`, margins, operating profitability,
- growth and valuation: `sale_growth`, `tobinsq`, `market_to_book`,
- financing and liquidity: leverage, cash, interest coverage,
- investment and composition: capital expenditures, R&D intensity, intangible assets,
- payout and repurchases,
- constraint indices such as **Whited-Wu** and **Hadlock-Pierce**.

This is more than a cosmetic merge. It creates the economic state variables needed to:
- control for confounding firm conditions,
- define heterogeneity splits,
- interpret M&A effects in a richer organizational context.

### Real scaling and CPI adjustment

The code downloads CPI data from FRED and uses it to build real assets and real R&D variables. That is another sign that the construction is meant to support serious empirical work rather than only descriptive charts.

### Linking Compustat to `permco`

The link to the patent side is done using the CRSP-Compustat link table. The code keeps primary, validated links and restricts them to valid date ranges.

```python
compustat_core_linked = compustat_core_df.merge(
    linktable[['gvkey', 'permco', 'linkdt', 'linkenddt']],
    on='gvkey', how='inner'
)
compustat_core_linked = compustat_core_linked[
    (compustat_core_linked['datadate_dt'] >= compustat_core_linked['linkdt']) &
    (compustat_core_linked['datadate_dt'] <= compustat_core_linked['linkenddt'])
]
```

This link discipline is essential. The project would be much weaker if the public-firm mapping were fuzzy or time-inconsistent.



## Step 6. Clean and link M&A deals

The M&A module transforms raw SDC transaction records into a deal panel that can be merged into firm-year and inventor-year data.

### Deal cleaning choices

Several choices are worth highlighting.

#### 1. Announcement timing is central
The project uses announcement-year timing for the event-study setup. That is reasonable because announcement is typically the point at which firms, investors, and employees begin responding to the transaction.

#### 2. Link both target and acquiror through CUSIP-6 and valid date windows
The code truncates CUSIPs to six digits and merges both sides of the deal through the link table. It then keeps only observations for which the announcement date falls within the valid link ranges for both firms.

#### 3. Restrict deal outcomes
The code keeps completed and withdrawn deals and explicitly creates a failed-merger indicator. This is useful because failed deals can later be used either as controls, robustness checks, or descriptive contrasts.

### Deal-level variables

The M&A panel retains:

- acquiror and target identifiers,
- transaction value,
- ownership percentages,
- diversifying indicators based on industry codes,
- deal outcome and failure status.

### Pre-deal technology similarity between target and acquiror

A particularly strong addition is the pre-deal technology similarity measure, built from five-year patent portfolios of the target and acquiror.

This is useful for at least two reasons:

1. it helps characterize whether deals combine related or more distant technology portfolios,
2. it provides a natural heterogeneity variable for later interpretation.

In other words, the M&A panel is not just a timing file. It is a rich deal-characterization file.



## Step 7. Assemble the final analysis panels

The final module turns the intermediate construction objects into the panels used for estimation.

This is the point where the project shifts from **data integration** to **econometric design**.

### 7.1 Firm-year panel

The code first aggregates patent outcomes to the `permco × year` level:

```python
firm_year_patent_metrics = pat_inv_firm_df.groupby(['permco', 'filing_year']).agg(
    total_patents=('patent_id', 'nunique'),
    cites=('cites', 'sum'),
    xi_real=('xi_real', 'sum'),
    ...
).reset_index()
```

It then merges in:

- rolling firm technology similarity,
- Compustat fundamentals,
- inventor arrival and departure measures,
- relative quality of arriving and departing inventors,
- M&A event counts and deal-value measures.

The result is a firm-year panel that simultaneously describes:
- innovative output,
- technology direction,
- labor flows,
- financial conditions,
- M&A exposure.

That integration is one of the most compelling aspects of the project.

### 7.2 Firm-year M&A event-study panel

The next object is a firm-year event-study panel centered on the **closest deal year** within a ±5-year window.

A nice engineering feature here is that the code avoids `merge_asof` and instead computes previous and next deal indices explicitly with `np.searchsorted`. That makes the logic more transparent and less fragile.

The panel then defines:

- `ma_deal_role` = acquiror, target, or no recent M&A,
- `years_from_ma_deal`,
- deal value,
- failed-merger flag,
- other-party identifier,
- pre-deal technology similarity.

This object is the main firm-level treatment panel used for DiD and event-study work.

### 7.3 Inventor-year panel

The inventor-year panel is built for inventors who ever patent at firms in the analysis universe. The code creates:

- annual inventor outcome measures,
- zero-filled years for no-patenting observations within the inventor career span,
- annual employer assignment,
- move-year indicators,
- M&A context of the assigned employer.

This is an ambitious and important design choice. It converts sparse patent observations into a panel suitable for longitudinal analysis of inventor behavior.

### 7.4 Inventor-event panel with matched controls

The most sophisticated construction step is the inventor M&A event-study panel.

The logic is:

1. identify treated inventor-deal events,
2. keep only events feasible over the full event window,
3. construct matched control pseudo-events using inventor characteristics at `t = -1`,
4. expand each event into a balanced `[-5, +5]` relative-year grid,
5. merge annual inventor outcomes back onto the event grid.

The matching features include variables such as:

- inventor age,
- cumulative patents,
- cumulative citations,
- first CPC subclass.

The code supports propensity-score-based matching with exact matching on selected categorical fields.

This is a strong design because it tries to build a credible inventor-level counterfactual while keeping the event-study object balanced and interpretable.

### Why these final panels are the right endpoint

By the end of the pipeline, the project has produced four conceptually distinct but connected research objects:

1. **firm-year panel** for aggregate organizational outcomes,
2. **firm event-study panel** for treatment-timing analysis,
3. **inventor-year panel** for individual-level longitudinal behavior,
4. **inventor-event panel** for matched event-study analysis.

That architecture reflects a PhD-level empirical strategy: it links micro mechanism evidence to firm-level outcomes rather than relying on only one level of aggregation.



## Construction choices that are especially strong, and a few places to be explicit about limitations

### What is especially strong in the construction

1. **The project builds real research infrastructure, not just a regression-ready CSV.**  
   The layered construction from raw sources to multiple final panels is one of the clearest signs of technical maturity in the repository.

2. **The measurement choices are economically motivated.**  
   Novelty, exploration, mobility, and technology similarity are not generic machine-learning features. They are tailored to the economic mechanisms of the question.

3. **The pipeline balances ambition with tractability.**  
   Intermediate caching, chunked citation routines, and explicit path validation show a practical understanding of what large empirical projects require.

4. **The final objects are designed for multiple estimators.**  
   The construction is clearly shaped by downstream DiD, event-study, and matching requirements.

### A few limitations or caveats worth stating openly

A strong notebook should also be explicit about where the construction is deliberately imperfect.

- **Public-firm scope.**  
  The use of `permco` is analytically powerful, but it means the project is strongest for the publicly listed-firm universe rather than the full economy of patent assignees.

- **Assignee-based employer inference.**  
  Employer assignment from patent assignees is sensible and standard in this setting, but it is still an inferred labor-market link rather than a direct HR record.

- **Strict move definitions trade off recall for precision.**  
  That is a feature, not a bug, but it should be stated clearly.

- **The split modules still rely on sequential shared-state execution.**  
  For a research repository, this is acceptable and often desirable for faithfulness. For a production software package, one would likely move further toward explicit function imports and typed interfaces.

### One issue I would state honestly

The setup file still contains a comment suggesting that some foundational economic-data loading is "assumed to be here for brevity." In the current split repository, those later steps are handled by subsequent modules. This does not appear to break the pipeline, but the notebook should describe the actual implemented sequence rather than the older monolithic-script commentary.

That is an example of the kind of small inconsistency worth acknowledging rather than hiding.



## Summary of the construction

The construction pipeline creates a research dataset that is substantially richer than a standard event-study input file.

### In one sentence

The project starts from raw patent, accounting, and deal records and builds a linked set of firm-level and inventor-level panels that can speak to both **organizational outcomes** and **microeconomic mechanisms** around M&A.

### What each stage contributes

- **Setup and orchestration** establish a reproducible research environment.
- **Patent panel construction** creates the core patent-inventor-firm microdata.
- **Quality and novelty measures** turn patents into economically meaningful innovation objects.
- **Exploration and exploitation metrics** capture the direction of technological search.
- **Mobility construction** identifies meaningful inventor moves and summarizes mover quality.
- **Technology similarity measures** help interpret whether movement reflects related or distant technological reallocation.
- **Firm fundamentals and link tables** anchor the patent side in standard corporate-finance measurement.
- **M&A linking** creates timed treatment events with deal characteristics.
- **Final panels** turn all of the above into objects suitable for DiD, event studies, heterogeneity analysis, and matched inventor-event designs.

### Why this matters for the broader project

For recruiters and economists at technology firms, the construction is important because it demonstrates a combination of skills that often appear separately but less often together:

- research design,
- large-scale data construction,
- identifier linking across sources,
- thoughtful feature engineering,
- sensitivity to measurement error,
- and an ability to structure the final output around a clear economic question.

That combination is a major part of what makes the project a credible showcase of independent empirical research skill.


## Analysis workflow: overview and design logic

The analysis side of the project is organized much more explicitly than the construction side. The repository-facing analysis code does **not** recreate the original monolithic script by sharing a live namespace across modules. Instead, it uses a cleaner split between configuration, data loading, reusable estimator functions, and two high-level runners:

- a **firm-panel branch**, and
- an **inventor-year / inventor-event branch**.

That choice is economically and computationally sensible. The firm-level and inventor-level designs answer related but distinct questions, use different units of observation, and have different memory bottlenecks. Keeping them separate makes the workflow easier to inspect and easier to explain to external readers.

At a high level, the logic is:

1. load the final panels produced by construction,
2. define the treatment and event-time structure,
3. run baseline two-way fixed-effects DiD and event studies,
4. layer on heterogeneity specifications,
5. run selected staggered-adoption robust estimators,
6. use placebo and balance diagnostics to decide what is credible,
7. interpret only the results that survive those checks.

That last step is important. The notebook should not present the analysis as “run many methods until something is significant.” It should present the analysis as **triangulation**: multiple estimators are used because the treatment is staggered, treatment effects can be heterogeneous, and simple TWFE event studies can be misleading.


## Analysis pipeline at a glance

The public workflow is intentionally split into small files that do one thing each.

### Core orchestration
- `run_analysis.py` calls the two main branches in sequence.
- `run_firm_panel_analysis.py` contains the full firm-level workflow.
- `run_inventor_year_analysis.py` contains the inventor-year workflow.

### Shared infrastructure
- `config.py` defines paths, windows, and global switches.
- `data.py` loads the cleaned panels and merges lagged firm controls into the inventor-level datasets.
- `utils.py` contains the reusable econometric helpers such as the two-way fixed-effects `PanelOLS` wrapper and the heterogeneity `Z` constructors.

### Topical method modules
- `firm_analysis.py` contains the baseline firm DiD/event-study workflow, the matched stacked design, and the generic triple-DiD/event-study machinery.
- `inventor_year.py` contains inventor-event preparation, within-firm heterogeneity splits, downsampling for advanced methods, and the inventor-year CSDID wrapper.
- `advanced_methods.py` contains the advanced estimators used as cross-checks: Causal Forest, Synthetic Control, Sun–Abraham, and Borusyak–Jaravel–Spiess.
- `placebos.py` contains lead and permuted placebo timing exercises.
- `summary_statistics.py` writes compact panel summaries and pre-period baselines.

This split is a strength of the repository version. It makes the analysis look like a serious research pipeline rather than a notebook that happened to work once.


## Step A. Configuration and loading of the analysis data

The analysis starts with a compact configuration object:

```python
@dataclass
class AnalysisConfig:
    analysis_window: tuple[int, int] = (1980, 2020)
    event_study_window: tuple[int, int] = (-5, 5)
    event_study_ref_k: int = -1
    run_inventor_year_advanced: bool = True
    advanced_alpha: float = 0.05
    inventor_year_csdid_bootstrap_draws: int = 30
```

This is good practice for a public research project. It puts the key design choices in one place:

- the **calendar window** for the usable panel,
- the **event-study window**,
- the omitted event-time reference period,
- and the switches controlling the more expensive advanced methods.

The data loader then reads the final construction outputs and attaches lagged firm controls to the inventor-side panels. That is a subtle but important design step. The inventor outcomes are naturally inventor-level, but the mechanism may still run through the characteristics of the employing firm. By merging lagged firm controls into the inventor panels, the project can ask whether inventor-level changes persist after accounting for the employer's pre-period scale and financial condition.

Conceptually, this means the analysis combines:
- **individual outcomes**,
- **firm event timing**, and
- **firm baseline controls**.

That is exactly the kind of multi-level empirical setup that signals strong applied micro / innovation-economics training.


## Step B. Firm-panel design: what the firm analysis is trying to estimate

The firm branch asks a relatively direct question:

> When a publicly listed firm becomes an acquiror or a target, how do its innovation and inventor-flow outcomes evolve relative to otherwise similar firms without a recent M&A event?

The raw event panel is first turned into a standard DiD/event-study input:

```python
did_df["Treated"] = (did_df["ma_deal_role"] != "no_recent_MandA").astype(int)
did_df["Post"] = (did_df["years_from_ma_deal"].fillna(-1) >= 0).astype(int)
did_df["Post_Treated"] = did_df["Treated"] * did_df["Post"]
did_df["gname"] = np.where(
    did_df["years_from_ma_deal"].isna(),
    0,
    did_df["data_year"] - did_df["years_from_ma_deal"],
).astype(int)
```

Economically, these objects do the following:

- `Treated` identifies firms exposed to an M&A event.
- `Post` marks the post-event period.
- `Post_Treated` is the standard average post-treatment effect regressor.
- `gname` stores the cohort year, which is essential for staggered-treatment methods.

The code then creates separate datasets for:
- **acquiror vs. no recent M&A**, and
- **target vs. no recent M&A**.

That separation is a very good design choice. Acquirors and targets should not be pooled mechanically. The organizational mechanisms are different:
- acquirors face integration, reallocation, and coordination decisions;
- targets face absorption, disruption, or reorganization from the receiving side.

Treating them as separate empirical objects gives the analysis a cleaner interpretation.


## Step C. Why the firm analysis uses a matched stacked design

The baseline firm analysis does **not** just run one pooled two-way fixed-effects regression on all firms. It first builds a matched stacked panel.

The matching function works cohort by cohort. At the year before treatment, it uses:
- industry (`sic3`),
- firm size (`log_sale`, `log_mv`),
- and a cohort-specific propensity score,

to pair treated firms with comparable controls. Within industry, the actual pair selection uses Mahalanobis distance on firm size variables, with an optional propensity-score caliper.

A compact summary of the matching logic is:

```python
for cohort_year in cohort_years:
    cov = df[df["data_year"] == (cohort_year - 1)].copy()
    pot = cov[(cov["gname"] == 0) | (cov["gname"] == cohort_year)].copy()

    # estimate cohort-specific propensity score
    lr.fit(X_ps, y_ps)
    pot["pscore"] = lr.predict_proba(X_ps)[:, 1]

    # within sic3, match on Mahalanobis distance in (log_sale, log_mv)
    dist = cdist(X_tr, X_ct, metric="mahalanobis", VI=VI)
```

This is attractive for three reasons.

### 1. It improves comparability
The design does not rely only on fixed effects to clean up large cross-sectional differences. It first tries to compare treated firms to firms that already looked similar before the event.

### 2. It respects treatment timing
Matching is done relative to each treatment cohort rather than once for the whole sample.

### 3. It makes the event-study interpretation sharper
Because the stacked panel is built around comparable treated-control sets for each cohort, the event-study plots are easier to interpret than a single fully pooled TWFE event study.

This is not a magic solution. Matching does **not** prove parallel trends. But it is a strong design enhancement, and for a recruiter or economist reading the notebook it signals that the empirical design is not naïve.


## Step D. Baseline estimation: fixed effects DiD and event studies

The core regression wrapper is intentionally simple and reusable. It runs a `PanelOLS` with:
- entity fixed effects,
- time fixed effects,
- clustered standard errors.

```python
mod = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
    check_rank=True,
)
```

For each outcome, the analysis does two related things:

### 1. Baseline DiD
Estimate the average post-treatment effect through `Post_Treated`.

### 2. Event study
Replace the single post indicator with event-time dummies interacted with treatment status, omitting `k = -1` as the reference year.

That baseline event-study machinery is important because it lets the notebook ask:
- whether effects build slowly or quickly,
- whether pre-trends are approximately flat,
- whether the treatment effect is concentrated in the deal year or after it.

The code also saves compact regression tables and coefficient plots for every outcome. That is good repo design because it makes the project reproducible and inspectable without forcing users to rerun everything.

### Caveat
The notebook should be explicit that plain two-way fixed-effects event studies can be biased under staggered timing and heterogeneous treatment effects. The project handles that appropriately by treating the baseline TWFE results as a **first screen**, not as the final word.


## Step E. Heterogeneity analysis: why the project uses `Z` interactions and triple-DiD-style specifications

A major strength of the project is that it moves beyond average effects. The analysis constructs several pre-treatment heterogeneity variables:

- baseline firm size (`Z_log_sales_cont`, plus quantile bins),
- deal size relative to market value (`Z_deal_rel_cont`, plus quantile bins),
- inventor rank within firm on cumulative patents or citations.

These are generated at the baseline period, typically `k = -1`, and then broadcast over the event window. That is the right way to do it. It keeps the heterogeneity split **predetermined** rather than allowing post-treatment variables to contaminate interpretation.

The generic event-study runner then supports a triple-DiD-style extension:

```python
df_use[post_z] = df_use["Post"] * z_num
df_use[triple] = df_use["Post_Treated"] * z_num
```

In plain language, this asks not only:

> Did treated firms change after the event?

but also:

> Did treated firms change **more or less** after the event depending on baseline size, deal scale, or inventor rank?

This is exactly the kind of heterogeneity design that makes the project look more mature. Tech economists and hiring managers are often less impressed by one average treatment effect than by a disciplined answer to **where** the effect is strongest and **why**.


## Step F. Advanced firm estimators: what each one contributes

The project then re-estimates selected outcomes with more specialized methods.

### 1. Sun–Abraham interaction-weighted event study
This is the most natural staggered-adoption robustness check for the baseline event studies. It avoids some of the weighting problems of TWFE event studies when treatment timing is staggered and treatment effects vary by cohort or over event time.

Why it matters:
- if a pattern is visible in both the matched TWFE event study and Sun–Abraham, confidence increases;
- if the shapes are very different, the TWFE result should be treated cautiously.

### 2. Borusyak–Jaravel–Spiess imputation estimator
This method imputes untreated outcomes and then constructs treatment effects from those imputed counterfactuals. It is a useful complement to Sun–Abraham because it relies on a different implementation logic.

Why it matters:
- agreement between BJS and Sun–Abraham is especially reassuring;
- disagreement is informative and should be discussed, not hidden.

### 3. Synthetic Control
The code runs firm-specific synthetic control fits and then aggregates relative gaps across successful treated units.

Why it matters:
- SCM is visually intuitive,
- it is useful for showing that some effects are not artifacts of a single regression functional form,
- and it gives a concrete unit-level interpretation.

Main caveat:
SCM is more fragile than it looks. Results depend on donor availability, pre-period fit, window choice, and whether a treated firm has enough support for a sensible synthetic counterpart.

### 4. Causal Forest
The firm analysis also runs a Causal Forest using pre-period covariates and short-run post outcomes.

This is **not** the main identification engine. It is best understood as a structured heterogeneity tool:
- which pre-period firm characteristics predict more negative or more positive treatment effects?
- does the forest agree with the simpler size-based heterogeneity splits?

That is the right way to present it. A forest can be very useful, but it should not be oversold as replacing the causal identification strategy.


## Step G. Placebo and balance checks: why they are central rather than decorative

A strong empirical notebook should make clear that credibility is built through failed tests as well as successful ones.

The analysis includes:

### Pre/post balance diagnostics
The matching design is evaluated by comparing treated and control firms on pre-period variables such as `log_sale` and `log_mv`, before and after the stacking procedure.

### Lead placebos
Treatment is artificially shifted earlier in time. If the model then finds a large effect before the actual event, that is evidence against a clean causal interpretation.

### Permutation placebos
Treatment timing is permuted across treated units. This checks whether the apparent signal is stronger than what one would expect from arbitrary timing assignments.

This is a very good robustness philosophy. It tells the reader:

- we are not just looking for significance,
- we are asking whether the effect looks specific to the actual event timing.

That stance is especially important if the project will be read by economists in tech or applied research teams, where skepticism about event-study designs is usually high.


## Step H. Inventor-year design: the mechanism side of the project

The inventor-year branch is where the project becomes especially distinctive. It asks a different question from the firm panel:

> What happens to inventors attached to treated firms, relative to matched control inventors, around merger events?

This is the more mechanism-oriented design. The firm panel tells us whether innovation or inventor flows changed at the organization level. The inventor panel tells us whether those firm-level changes are associated with:
- inventor mobility,
- changes in exploration versus exploitation,
- and changes in individual inventive output.

The workflow cycles over:
- role = `acquiror` or `target`,
- whether lagged firm controls are included,
- whether inventor-side controls are included,
- and several heterogeneity layers.

That branching structure is not accidental. It is a way of checking whether the substantive conclusions survive moderate changes in conditioning information.


## Step I. Preparing the inventor event panel: treated inventors versus matched controls

The inventor panel is built from the balanced inventor-event-study panel created in construction. The analysis then extracts a role-specific sample such as “acquiror inventors vs. control anchors” or “target inventors vs. control anchors.”

A central preparation step is:

```python
df = df[(df["ma_deal_role"] == role_l) | (df["is_control_event"] == 1)].copy()
df["Treated"] = (df["ma_deal_role"] == role_l).astype(int)
df["event_time"] = pd.to_numeric(df["years_from_ma_deal"], errors="coerce").astype(float)
df["cohort"] = pd.to_numeric(df["closest_deal_year"], errors="coerce").astype(float)
df["Post"] = (df["event_time"] >= 0).astype(int)
df["Post_Treated"] = df["Post"] * df["Treated"]
```

The identification logic is therefore parallel to the firm panel, but the unit of analysis is now an inventor-event rather than a firm-year.

A particularly strong feature is that the code can also create **within-firm rank indicators** based on cumulative patents or citations at `t = -1`. That makes it possible to ask whether higher-ranked inventors respond differently to M&A than lower-ranked inventors within the same firm context.

That is a very appealing design feature for a job-market or recruiter-facing notebook because it shows attention to the internal composition of firms, not just the average inventor.


## Step J. Baseline inventor outcomes and why they matter

The inventor-year branch focuses on a deliberately interpretable set of outcomes:

- `total_patents`
- `cites`
- `xi_real`
- `novelty_score_group`
- `exploration_inv`
- `exploitation_inv`
- `is_move_year`

This is a good outcome list. It spans three conceptually different margins:

### 1. Mobility
`is_move_year` is a direct mechanism outcome. If M&A affects organization, incentives, or match quality, inventor moves are one of the most natural places to look.

### 2. Direction of search
`exploration_inv` and `exploitation_inv` speak to whether inventors stay in familiar technology areas or move into newer ones.

### 3. Individual output and influence
Patents, citations, and `xi_real` capture different notions of inventor productivity and impact.

This mix is more persuasive than relying only on patent counts. It shows the reader that the project is not treating innovation as a one-dimensional object.


## Step K. Inventor-year advanced methods and why the project uses them selectively

The inventor panel is much larger and more computationally demanding than the firm panel, so the advanced methods are used more selectively.

### 1. CSDID / Callaway–Sant'Anna
The code runs a compact wrapper around `ATTgt`, using role-specific cohort definitions such as:
- `cs_g_year_target`,
- `cs_g_year_acquiror`,
- `cs_g_year_all`.

This is a strong choice. Inventor exposure to treatment is genuinely staggered, and a cohort-based estimator is appropriate. The code also trims weakly identified cells and keeps the post horizon modest, which is exactly what one should do when the panel is large and some cohorts are thin.

### 2. Sun–Abraham
For outcomes that are significant in the baseline inventor specification, the code reruns Sun–Abraham as a robustness check on the inventor-event panel.

### 3. Optional BJS
The inventor workflow is set up so that BJS can also be run, but the public configuration keeps this limited because of runtime and memory considerations.

### 4. Downsampling for advanced methods
The project explicitly downsamples inventor-event units before running the heavier estimators.

That is a sensible engineering compromise, not a conceptual weakness, as long as the notebook is honest about it. The right way to present it is:

- baseline DiD/event-study uses the full prepared panel;
- advanced methods are used on a carefully downsampled but balanced subset to keep the workflow feasible.

That explanation reads as practical and credible.


## Step L. What the current results appear to say

This part should be presented carefully because I do **not** have the exported result tables in this upload. The discussion below is therefore based on the current notebook draft, the active analysis code, and the earlier project discussions rather than on a fresh read of the saved CSV outputs.

With that limitation stated explicitly, the most credible interpretation still seems to be:

### 1. The clearest evidence is on the inventor / mechanism side
The project appears strongest when it studies **mobility** and **exploration** rather than trying to make a blanket claim about universal output decline.

That is attractive intellectually and strategically:
- it is more distinctive,
- it is closer to the actual micro mechanism,
- and it is easier to defend if aggregate output measures are mixed.

### 2. Acquiror-side effects look more coherent than target-side effects
Earlier runs and notes suggest that the acquiror side often produces cleaner and more interpretable patterns than the target side.

That makes economic sense. Acquirors are where integration, reallocation, and coordination decisions are actively made.

### 3. Exploration is more useful than exploitation as a headline
Because exploitation is mechanically close to the mirror image of exploration in this setup, exploration is usually the stronger variable for presentation. It carries the mechanism more naturally and avoids redundancy.

### 4. Heterogeneity by size seems substantively informative
The size-based `Z` interactions appear to matter. The emerging story is not simply “M&A hurts innovation,” but rather that the effect depends on the firm's pre-period capacity and perhaps on deal scale relative to firm size.

That is a much better research contribution than a generic average-effect statement.


## Step M. How I would present the strongest results in the final notebook

Based on the code structure and the earlier discussions, I would recommend presenting the findings in the following hierarchy.

### Headline finding
**M&A reshapes inventive organization more clearly through inventor mobility and changes in exploratory search than through a uniform collapse in output.**

### Supporting finding 1
**The acquiror side shows the cleanest post-event changes.**

### Supporting finding 2
**Firm size moderates the post-deal response.**

### Supporting finding 3
**Aggregate patent and citation effects are more mixed and should be treated cautiously.**

That ordering is important. It makes the project sound like a serious innovation-economics paper rather than an overclaimed “M&A kills innovation” story. For recruiting purposes, that is an advantage: the project reads as careful, mechanism-aware, and empirically disciplined.


## Step N. What the notebook should say explicitly about limitations

A strong public notebook should make the following limits visible.

### 1. Public-firm scope
The analysis follows inventors only when they can be linked to publicly listed firms. That improves observability but means the mobility results are not the full universe of inventor moves.

### 2. Matching improves design but does not prove exogeneity
The matched stacked design is a strong improvement over a raw pooled comparison, but it does not eliminate all concerns about differential pre-trends or event selection.

### 3. Staggered-adoption methods help, but they are still design-based estimators
Sun–Abraham, BJS, and CSDID reduce known TWFE problems, but they do not make interpretation mechanical. One still has to inspect support, pre-trends, and estimator agreement.

### 4. Advanced inventor methods are run on reduced samples
This is a practical choice to keep the workflow feasible, but it means the most computationally intensive cross-checks are not literally estimated on every row of the full panel.

### 5. Output effects are less settled than mechanism effects
The project should not oversell patent counts or citations if those outcomes are less stable across methods.

Writing these caveats directly into the notebook will improve trust rather than weaken the project.


## Summary of the analysis

The analysis workflow is one of the strongest parts of the project.

It combines:

- a careful **firm-level matched stacked event-study design**,
- an **inventor-level mechanism design** built around matched control pseudo-events,
- multiple **staggered-adoption robust estimators**,
- disciplined **heterogeneity analysis** using baseline `Z` variables,
- and a good set of **placebo and balance diagnostics**.

Just as important, the code structure reflects good empirical judgment. The project does not treat all statistically significant outputs as equal. It distinguishes between:
- baseline screens,
- robustness checks,
- and results that are actually persuasive enough to headline.

For external readers, that is exactly the right signal. It shows not only coding ability, but also the applied-economics skill of deciding **which evidence should matter most**.


## Empirical design in plain language

The project uses publicly listed firms and patent-linked inventors to study what happens around merger and acquisition events.

At a high level, there are two complementary designs:

### 1. Firm-year panel
This asks whether treated firms differ from comparable untreated firms after a deal. It is useful for aggregate outcomes such as:
- patent counts,
- citations,
- novelty and exploration measures,
- inventor flows in and out of the firm.

### 2. Inventor-year / inventor-event panel
This asks whether inventors attached to treated firms behave differently after a deal relative to matched controls. This is especially useful for:
- mobility,
- exploration,
- inventor-level productivity,
- within-firm heterogeneity.

The value of the project is that it combines both levels:
the firm panel captures organizational outcomes, while the inventor panel helps interpret the mechanism.

## Identification strategy

The main estimators are:
- baseline fixed-effects DiD,
- event studies,
- Sun–Abraham interaction-weighted event studies,
- Borusyak–Jaravel–Spiess imputation,
- selected matching and stacked-panel designs.

The reason for using multiple estimators is straightforward: staggered treatment timing can make simple two-way fixed-effects event studies misleading, especially when treatment effects evolve over time or differ across cohorts.

That is why, in interpretation, this notebook gives more weight to:
- patterns that survive across multiple estimators,
- results without severe pre-trend problems,
- economically sensible magnitudes and shapes.

It gives less weight to:
- isolated significant coefficients,
- event studies with visibly unstable pre-trends,
- very large or implausible coefficients,
- patterns that reverse sign across methods without a clear explanation.

In [ ]:
# Core imports for the notebook
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Repository root: adjust after cloning if needed
REPO_ROOT = Path(".").resolve()

# Example output folders used by the analysis script
MODEL_OUT = REPO_ROOT / "Model_outputs"
PLOTS_OUT = MODEL_OUT / "Plots"

print("Repository root:", REPO_ROOT)
print("Model outputs:", MODEL_OUT)

## Step 1. Load the key outputs

The notebook is easiest to use if the heavy analysis has already been run and the output CSVs / plots already exist.

The helper file added to the repository can build:
- summary statistics tables,
- panel diagnostics,
- robustness scorecards,
- compact headline tables for the notebook.

The next cell assumes those helper functions are available.

In [ ]:
# These imports assume the helper file from this draft has been added to the repo.
# If the file is placed elsewhere, adjust the import path.
try:
    from notebook_additions_minimal import (
        build_headline_scorecard,
        summarize_firm_panel,
        summarize_stacked_panel,
        summarize_inventor_event_panel,
        build_result_inventory,
        write_notebook_support_outputs,
    )
    HELPERS_AVAILABLE = True
except Exception as e:
    HELPERS_AVAILABLE = False
    print("Helper import failed:", e)

## Step 2. Recommended summary statistics to include

A public-facing notebook should show basic descriptive structure before jumping into causal estimates.

I recommend including four compact tables:

1. **Firm panel summary**  
   Number of firms, years, treated firms, and baseline means for headline outcomes.

2. **Stacked firm-panel summary**  
   Number of treated and control firm-event units, balance over the event window, and treated share.

3. **Inventor-event panel summary**  
   Number of inventors, inventor-event units, treated and control units, and event-window balance.

4. **Headline baseline means**  
   Means at `t = -1` for the outcomes that appear in the narrative.

This helps readers anchor effect sizes and see the scale of the analysis.

In [ ]:
# Example usage once the relevant dataframes are loaded in memory:
#
# firm_summary = summarize_firm_panel(firm_panel, outcomes=["xi_real", "arriving_inventors_count", "exploration_firm"])
# acq_stack_summary = summarize_stacked_panel(stacked_acquiror, label="Acquiror stack")
# tgt_stack_summary = summarize_stacked_panel(stacked_target, label="Target stack")
# inv_summary = summarize_inventor_event_panel(inv_es, label="Inventor event panel")
#
# display(firm_summary)
# display(acq_stack_summary)
# display(tgt_stack_summary)
# display(inv_summary)

## Step 3. How to think about the results

A good empirical reader should evaluate the results in layers rather than asking whether one coefficient is statistically significant.

### A. What is a true headline result?
A result is headline-worthy if most of the following are true:
- the sign is stable across reasonable specifications,
- event-study pre-trends are not obviously problematic,
- the magnitude is economically interpretable,
- the estimate aligns with the mechanism story,
- more robust staggered-treatment estimators do not overturn it,
- and placebos do not generate similarly strong patterns.

### B. What is a supporting result?
A result is useful but secondary if:
- it supports the mechanism,
- it fits the economic story,
- but it is weaker, noisier, or more sensitive than the main result.

### C. What should stay out of the headline?
Results should stay out of the headline if:
- they are driven by a single estimator,
- they rely on implausibly large coefficients,
- they have visibly non-flat pre-trends,
- they reverse sign across methods without a clear explanation,
- or they are only present in a downsampled advanced run and not in the full baseline panel.

This project is strongest when interpreted with that discipline.



## Executive synthesis of the empirical results: firm panel and inventor-year panel together

Before turning to the estimator-by-estimator discussion, it helps to state the overall message of the project in plain research terms.

At a high level, the evidence points to **post-merger disruption with strong heterogeneity**, not to one clean universal average treatment effect. That statement is true in both the **firm panel** and the **inventor-year panel**, although the two panels illuminate different margins.

### 1. What the firm panel says

The firm panel answers the aggregate question: **how does the innovative output of treated firms evolve after M&A relative to matched controls?**

Across the baseline stacked DiD specifications, the first-pass picture is fairly negative. Both acquirors and targets often show weaker post-treatment patenting, citation-based outcomes, and innovation-value measures. That is exactly the kind of pattern one would expect if mergers interrupt project pipelines, create internal overlap, delay decisions, or trigger selective departure of inventors and managers.

But the more advanced estimators make clear that the average-effect story is **not equally strong for every outcome**:

- For **acquirors**, the more convincing average-effect evidence is concentrated in outcomes such as **cited patents** and some related patent-composition measures, rather than in every patent bucket or every inventor-flow margin.
- For **targets**, the average-effect results tend to look negative as well, but the uploaded advanced evidence is thinner and sometimes noisier than for acquirors.
- The most stable pattern in the firm panel is **heterogeneity**. Larger acquirors appear more resilient, while on the target side the strongest negative effects tend to be associated with **larger deals relative to the target**.

Economically, that is a strong and intuitive result. A merger is not just a scale shock. It is an organizational shock. Firms with deeper managerial capacity, broader internal labor markets, and more slack are better able to absorb it. Smaller or more exposed firms are more likely to see integration frictions translate into weaker observed innovation.

### 2. What the inventor-year panel says

The inventor-year panel answers a different question: **how do individual inventors associated with treated firms change their behavior and outcomes after the deal?**

This is not just a smaller-scale version of the firm panel. It is a mechanism panel. It tells us whether the post-merger shock looks like:

- lower inventor productivity,
- greater inventor mobility,
- a shift from exploitation toward exploration,
- selective retention of some inventors but not others,
- or simple reallocation across project types.

That distinction matters. A firm can lose aggregate output even when some inventors become more exploratory, and a firm can preserve inventor-level activity while still losing organization-level efficiency because coordination breaks down.

The baseline inventor-year results suggest a **sharp contrast between acquirors and targets**:

- For **acquirors**, the baseline results look like **reorganization**. Inventors appear to become more exploratory, less exploitative, slightly more novel, and somewhat more likely to move, while measures such as `xi_real` deteriorate.
- For **targets**, the inventor-year baseline looks more uniformly negative: weaker patents, weaker citations, lower novelty, lower `xi_real`, and weaker move-related outcomes.

That contrast is economically plausible. Acquiror inventors remain inside the surviving organization and may be reallocated across technologies, teams, and product lines. Target inventors are more likely to face dislocation, loss of fit, or reduced access to the organizational resources that supported their pre-deal productivity.

### 3. Why the firm and inventor panels do not have to match one-for-one

It would actually be surprising if the firm panel and inventor panel were numerically identical in sign and magnitude.

There are several reasons:

1. **Aggregation changes the object being measured.**  
   Firm outcomes combine many inventors, units, and continuation choices. Inventor outcomes isolate individual behavior.

2. **Selection matters.**  
   Post-merger firms may retain some inventors, lose others, and reassign others. The firm panel combines those margins; the inventor panel partially separates them.

3. **Innovation is multi-dimensional.**  
   Patent counts, citations, novelty, and `xi_real` are not the same object. A merger may raise search and recombination while lowering realized value in the short run.

4. **Timing matters.**  
   Inventor behavior can adjust quickly. Firm-level citation outcomes adjust slowly because they are tied to filing, grant, and citation lags.

### 4. Econometric intuition for why the results are mixed across estimators

The variation across estimators is not a flaw by itself. In this setting, some disagreement is exactly what one should expect.

- **Stacked DiD** is a useful benchmark, but in staggered-adoption settings with dynamic effects it can still blur timing heterogeneity.
- **Sun–Abraham** is especially useful for seeing whether apparent post-treatment effects are partly reversals of pre-trends.
- **BJS** is attractive because it is robust to treatment-effect heterogeneity under its identifying assumptions, but it is still sensitive to support and to how outcomes are measured.
- **CSDID** is valuable for dynamic aggregation in staggered settings, but in these inventor-year outputs some of its estimates are extremely precise and occasionally at odds with both baseline and SA, which is a sign to interpret them as informative rather than automatically decisive.
- **SCM** can be appealing for selected firm outcomes, but successful fit counts matter, and in sparse patent outcomes the subset of units with usable donor fits can differ meaningfully from the full treated sample.
- **Causal forests** are not delivering sharp average treatment effects here, but they are useful because they repeatedly identify baseline firm size and related financial characteristics as important moderators.

In other words, the right use of the estimator set is not “vote counting.” It is triangulation.

### 5. What results should anchor the notebook

If the notebook is meant to be clear, persuasive, and useful for recruiting, the results hierarchy should be:

**Headline results**
- M&A is associated with meaningful post-merger innovation disruption.
- The disruption is **heterogeneous**, not uniform.
- Larger acquirors appear more resilient.
- Targets in relatively larger deals appear more exposed.
- On the inventor side, acquirors look reorganized; targets look more clearly disrupted.

**Supporting mechanism results**
- Acquiror inventor-year outcomes suggest a shift toward exploration and away from exploitation.
- Acquiror patent-composition results suggest that externally recognized patent output may deteriorate more clearly than raw counts alone imply.
- Mobility outcomes point to reallocation and selective reshuffling, though they are not equally stable across methods.

**More fragile or secondary results**
- Outcomes with mixed signs across advanced estimators,
- outcomes with evident placebo sensitivity,
- and outcomes based on narrow patent buckets or short-support dynamic cells.

That framing makes the project stronger, not weaker. It shows that the analysis is taking both economics and identification seriously.



## Results discussion based on the exported outputs in this upload

This section now reflects the broader set of files provided across both uploads:

- baseline and triple-DiD significance summaries for **acquiror** and **target** firms,
- selected **Borusyak–Jaravel–Spiess (BJS)** event-study outputs,
- selected **Sun–Abraham (SA)** event-study outputs for acquiror outcomes,
- selected **stacked synthetic control (SCM)** summaries,
- selected **causal forest (CF)** summaries,
- and **CSDID feasibility diagnostics** for selected acquiror outcomes.

The goal of this discussion is not to force every estimator into a single sign. The stronger goal is to identify which findings are **stable enough to anchor the main story**, which findings are useful as supporting mechanism evidence, and which findings should be presented more cautiously.

A core theme running through the outputs is that this is **not** a setting where one should expect every estimator and every outcome to line up perfectly. Firm-level patenting is lumpy, citations arrive with delay, many outcomes are zero-heavy, and mergers can alter both real innovative activity and the internal organization of patent production. That makes this an excellent setting for a serious applied-economics notebook: the right task is to separate **robust structure** from **estimator-sensitive averages**.

Two broad conclusions remain the safest and strongest.

1. The evidence does **not** support a simplistic universal claim such as “M&A always lowers innovation by X percent.”
2. The evidence **does** support a richer conclusion: merger effects are heterogeneous, and much of the action appears to come through **integration capacity, portfolio reorganization, and differential ability to absorb disruption**.



## What looks strongest in the current evidence

### 1. Baseline DiD still provides the useful first-pass picture
The baseline stacked DiD continues to suggest that post-merger periods often look weaker on standard innovation outcomes.

For **acquirors**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_uncited_patents`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- `log1p_self_cites`,
- `log1p_top10_to_2_patents`,
- `log1p_arriving_inventors_count`,
- `log1p_departing_inventors_count`,
- and `log1p_sum_patents_pre_move_arrivals`.

For **targets**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_cites`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- and `avg_rel_exploration_departures`.

This remains useful because it says that, before we get sophisticated, the data already look consistent with **post-deal disruption**. In firm-level innovation data, that is economically plausible. Mergers can interrupt project pipelines, create overlapping R&D teams, trigger inventor exits, delay internal decision-making, and redirect managerial attention away from exploratory activity.

At the same time, baseline DiD is not the end of the story. In staggered-treatment settings with noisy firm outcomes, a strong notebook should immediately ask:
- do more robust event-study estimators tell a similar story,
- do the signs hold for the more interpretable outcomes,
- and do the patterns fit an economic mechanism rather than a purely statistical artifact?

That is where the uploaded SA, BJS, SCM, and CF results become especially informative.


## The best public-facing headline: heterogeneity is the main result, not a uniform average effect

The firm-level results are now sufficiently developed to support a disciplined headline: **the central result is heterogeneity in post-merger innovation performance, not one uniform average effect of M&A.** The average firm-level effects are often negative or mixed, but the economically clearer and more reproducible pattern is that merger effects depend strongly on the kind of firm absorbing the shock.

### Acquirors: baseline size is the strongest and most coherent moderator

For acquirors, the most convincing heterogeneity dimension is still **baseline firm size**, proxied by pre-deal log sales. The newly uploaded baseline `Z_log_sales_q3` files reinforce rather than weaken that conclusion. In the low-sales tercile, the baseline post-merger coefficients are often negative or close to zero for core outcomes, while the interaction terms for larger terciles are frequently positive for economically important margins such as `log1p_total_patents`, `log1p_cites`, `log1p_cited_patents`, `log1p_uncited_patents`, `log1p_arriving_inventors_count`, and `log1p_xi_real`. The broad pattern is therefore not that large acquirors experience large positive post-merger gains in absolute terms, but that they appear **substantially more resilient than smaller acquirors when integration begins to disrupt innovation**.

That interpretation is economically intuitive. Larger acquirors likely have:
- deeper managerial capacity to handle integration,
- more diversified R&D portfolios,
- stronger internal labor markets,
- more slack to preserve projects during restructuring,
- and greater absorptive capacity when teams, technologies, and reporting lines are recombined.

In that sense, scale does not eliminate merger friction, but it appears to **attenuate the damage**. This is exactly the kind of heterogeneity pattern that is more interesting than a single average-treatment estimate, because it points to a concrete organizational mechanism.

### Targets: deal intensity matters more than simple size

For targets, the most compelling heterogeneity result remains **deal value relative to target scale**. The notebook already emphasized this, and nothing in the later uploads overturns it. The target-side story is therefore naturally complementary to the acquiror-side story:

- for **acquirors**, what matters most is baseline organizational capacity;
- for **targets**, what matters most is how transformative the transaction is relative to the target itself.

That asymmetry is economically plausible. Acquirors are the integrating organizations, so baseline managerial and technological depth should matter most on their side. Targets, by contrast, are more likely to experience disruption in proportion to how large and consequential the deal is relative to their own scale.

### Why the contrast with relative deal value for acquirors is substantively useful

The newly uploaded acquiror specifications based on **relative deal value** remain much weaker overall. Outside of a few organizational margins such as inventor inflows, headcount-related measures, and rolling self-similarity, they do not show a broad or stable gradient across core patent or citation outcomes. That contrast is substantively useful. It suggests that on the acquiror side, the key issue is not simply whether the deal is large relative to the balance sheet or market value. The more important question is whether the acquiror has the **organizational capacity to absorb and govern the integration shock**.

So the firm-level public-facing headline can now be stated more sharply: **M&A effects are heterogeneous in a way that maps naturally into organizational economics. Large acquirors are better able to absorb disruption, while larger relative deals are more disruptive for targets.**


## How the advanced estimators change the interpretation

The additional files sharpen the interpretation in two ways.

### A. Average effects are real enough to be interesting, but not always stable enough to be the sole headline
For the outcomes where more robust estimators were uploaded, the signs are often suggestive but not perfectly aligned across methods.

#### Acquiror inventor-flow outcomes
For **arriving inventors**:
- baseline DiD is negative,
- BJS is negative in post years 1 through 3 and remains negative later, though less precise,
- SCM is near zero overall,
- and the causal forest ATE is positive but very imprecise.

For **departing inventors**:
- baseline DiD is negative,
- BJS is positive but imprecise,
- SCM is negative and statistically significant,
- and the causal forest ATE is again imprecise.

The right interpretation is not that one method is “wrong” and another “right.” These outcomes are measuring a reorganization margin that can move in complicated ways. A merger can simultaneously reduce net inflows, selectively retain key staff, and create exits among overlapping or lower-priority teams. Depending on the identifying comparison and aggregation, different methods can emphasize different pieces of that adjustment.

#### Target citations
For **target citations**, the cross-method disagreement remains substantial:
- baseline DiD is negative,
- BJS is positive in the uploaded post-treatment years,
- SCM is negative and only borderline precise,
- and the causal forest ATE is close to zero with a very wide interval.

This is exactly the kind of outcome where caution is warranted. Citation outcomes are particularly noisy because they are affected by both the flow of new patents and the ex post citation process, which arrives with delay and can be influenced by portfolio composition, not just contemporaneous research productivity.

### B. The patent-composition outcomes for acquirors add a useful new layer
The newly uploaded acquiror files make the story richer. They suggest that the post-merger response is not only about “more or fewer patents,” but also about **what kind of patents and citation activity move most clearly**.



## New insight from the additional acquiror files: composition matters, not just counts

The new acquiror outputs for `cited_patents`, `top10_to_2_patents`, `uncited_patents`, and `self_cites` are especially helpful because they speak to **composition of innovative output**.

### 1. Cited patents look more consistently negative than several other patent components
For **acquiror cited patents**:
- baseline DiD is negative,
- BJS is negative in every uploaded post-treatment year and statistically meaningful early on,
- SCM is clearly negative and statistically significant overall,
- while the causal forest ATE is imprecise.

This is one of the cleaner pieces of evidence that post-merger disruption may be concentrated in **patents that attract downstream attention**, rather than only in raw patent volume.

A reasonable economic interpretation is that integration shocks disproportionately affect the kinds of projects that require coordination, continuity, and time to mature into influential patents. Cited patents are often produced by stronger projects or more effective internal execution. Those may be exactly the projects most vulnerable to disruption when teams are restructured or decision rights are reassigned.

### 2. Mid-tier cited patents also lean negative
For **top10_to_2_patents**, the evidence is somewhat weaker than for cited patents overall, but still leans negative:
- baseline DiD is negative,
- SA drifts negative after treatment,
- SCM is negative and statistically significant,
- BJS is also negative though not very precise,
- and the causal forest ATE is imprecise.

This is useful because it suggests the pattern is not confined only to the most extreme citation tail. It looks more like a broader weakening in the distribution of externally recognized patent output.

### 3. Uncited patents are harder to interpret and probably noisier
For **uncited patents**, the methods disagree more:
- baseline DiD is negative,
- SA turns clearly negative after treatment and becomes more negative over event time,
- SCM is strongly negative,
- but BJS is positive though imprecise,
- and the causal forest ATE is near zero and imprecise.

This is not surprising. Uncited patents are a particularly noisy object. They may represent genuinely low-impact inventions, patents that have not had time to accumulate citations, defensive filings, or simply sparse outcomes in firm-years with many zeros. Because of that, uncited-patent results should be treated as **supportive texture**, not the center of the story.

### 4. Self-citations are conceptually ambiguous
For **self-cites**, the estimates are again mixed:
- baseline DiD is negative,
- SA becomes negative in later post years,
- SCM is negative overall,
- but BJS turns positive and grows over post-treatment years, though imprecisely,
- and the causal forest ATE is positive but wide.

This is a case where economic interpretation matters a lot. Self-citations can fall if integration disrupts internal knowledge continuity. But they can also rise if the combined firm consolidates overlapping technological lines and reuses its own portfolio more intensively. In other words, self-cites are not a clean “good” or “bad” innovation measure. They partly reflect how the merged firm reorganizes internal technological inheritance. That is exactly why they should be discussed, but not used as a headline metric.


## Additional discussion: acquiror triple-DiD event studies by baseline sales terciles

The newly uploaded files add an important dynamic layer to the **acquiror-side triple-DiD event studies** that interact treatment timing with baseline sales terciles, `Z_log_sales_q3`. These results are worth discussing explicitly because they show why the heterogeneity story is stronger than the pooled event-study story.

### How to read these event-study tables

In these specifications, the coefficients `rt_k` trace the dynamic treatment effect for the **omitted sales tercile**, which is most naturally interpreted as the **lowest baseline sales tercile**. The interaction terms `rt_k__Z_log_sales_q3_1` and `rt_k__Z_log_sales_q3_2` are deviations for the middle- and high-sales terciles relative to that omitted low-sales group.

So for any event year `k`:

- `rt_k` is the effect for the low-sales tercile,
- `rt_k + rt_k__Z_log_sales_q3_1` is the effect for the middle-sales tercile,
- `rt_k + rt_k__Z_log_sales_q3_2` is the effect for the highest-sales tercile.

That is crucial. The raw `rt_k` path by itself can make the dynamics look more negative than they are for larger acquirors, because it shows only the lowest-sales firms directly.

### High-level pattern across the uploaded acquiror files

Across the uploaded sales-tercile event studies, a common pattern emerges.

First, the **lowest-sales acquirors** often display the most problematic dynamics. In several outcomes they have positive pre-treatment coefficients relative to the omitted `t = -1` period and then flatten or drift negative after treatment. This is especially visible in outcomes such as `log1p_cites`, `log1p_total_patents`, and `rolling_self_similarity`. That means the lowest-sales firms are precisely where pre-trend concerns are most pronounced and where post-merger weakening appears strongest.

Second, the **middle- and high-sales terciles usually look better than the low-sales tercile**, often substantially so. The interaction terms are frequently positive for core outcomes, especially for the highest tercile, implying that larger acquirors experience a materially smaller deterioration than small acquirors and in some cases remain close to zero on net.

Third, the dynamic evidence is most persuasive for a **relative-resilience interpretation**, not for a clean “big firms improve after mergers” claim. The event-study paths do not show a universal post-merger boom among large acquirors. Rather, they show that the adverse dynamic visible among smaller acquirors is **muted for larger acquirors**.

### What this adds beyond the pooled baseline event studies

This is why the triple-DiD event studies are so valuable. The pooled event studies average together firms with very different capacities to absorb disruption. Once the sample is allowed to vary systematically with baseline scale, the firm-level results become easier to interpret:

- small acquirors look more vulnerable to integration-related disruption,
- large acquirors look more capable of preserving innovative activity,
- and the resulting average effect is an amalgam of those very different paths.

That is a more publication-ready interpretation than simply reporting that “the average post coefficient is negative.” It links the empirical pattern to a concrete economic idea: **absorptive capacity and organizational depth shape whether a merger becomes mainly disruptive or merely manageable.**

## Additional discussion: acquiror triple-DiD by relative deal value

The newly uploaded **acquiror-side triple-DiD specifications by relative deal value**, `Z_deal_rel_q3`, are useful mainly because they clarify what the data are **not** strongly saying. In contrast to the sales-based heterogeneity results for acquirors, these specifications do **not** generate a broad, stable pattern across patent counts, citation measures, or the exploration/exploitation outcomes. That negative result is itself informative. It suggests that, on the acquiror side, **baseline organizational capacity and scale** matter more consistently than deal size relative to the acquiror’s own market value.

The newly uploaded **baseline results for the same `Z_deal_rel_q3` interaction** reinforce that interpretation rather than changing it. Across the baseline files, the relative-deal-value interaction remains mostly weak for the core patent and citation outcomes. The few suggestive margins are concentrated in organizational or mechanism-style outcomes such as inventor arrivals, selected mobility-quality measures, and rolling self-similarity. Even the level specifications for `exploration_firm` and `exploitation_firm` remain close to zero. So the broader conclusion is now more secure: this is not simply a feature of one transformed outcome or one event-study variant. In the acquiror sample, **relative deal value is not emerging as a strong general moderator of post-merger innovation performance**.

### How to read these triple-DiD tables

As in the other grouped heterogeneity specifications, the omitted category is most naturally interpreted as the **lowest relative-deal-value tercile**. Under that coding:

- `Post_Treated` is the post-merger treatment effect for the omitted low relative-deal group,
- `Post_Treated_Z_deal_rel_q3_1.0` is the incremental post effect for the middle tercile relative to that omitted group,
- `Post_Treated_Z_deal_rel_q3_2.0` is the incremental post effect for the highest tercile relative to that omitted group.

### What the results imply

The most natural reading is therefore comparative. For acquirors, the data are not telling a story in which “bigger relative deals uniformly hurt more” or “bigger relative deals uniformly help more.” Instead, the relative-deal-value split seems to matter only sporadically, and mostly on secondary margins. That is substantively useful because it sharpens the main claim:

- **acquiror heterogeneity is primarily about absorptive capacity and organizational scale**;
- **target heterogeneity is more about deal intensity relative to the target**.

That asymmetry gives the project a cleaner economic interpretation than a symmetric deal-size story would. It suggests that the burden of integration is not governed by the same variable on both sides of the transaction.

## Additional discussion: baseline acquiror event-study results and how they compare with the triple-DiD specifications

The newly added baseline event-study outputs are useful because they answer a different question from the sales-heterogeneity specifications. The baseline models ask whether, **on average**, acquiror firms exhibit systematic changes around the deal. The triple-DiD models then ask whether those pooled average effects conceal meaningful differences across acquirors with different baseline sales.

### 1. What is most stable across the baseline firm-panel event studies?

The baseline results now point to a fairly consistent qualitative pattern, but one that needs to be stated carefully. On average, acquiror firms do **not** show a dramatic immediate collapse at the event year. Instead, the most common pattern is:

- modestly elevated or flat coefficients in the pre-period,
- little clear break exactly at treatment,
- and a gradual move toward weaker outcomes in the later post years for several patent and citation measures.

That pattern is especially visible in `log1p_cites`, `log1p_total_patents`, `log1p_uncited_patents`, and `rolling_self_similarity`. In each of these cases, the later post-treatment years look weaker than the omitted pre-period, although the timing and precision differ across outcomes.

This is economically plausible. Integration frictions need not show up immediately in annual patent production. Some projects are already in the pipeline when the merger closes, while the effects of reorganization, reprioritization, and team disruption can accumulate over several years.

### 2. What do the baseline event studies say about pre-trends?

The baseline event studies also make clear that **pre-trends are not equally clean across outcomes**.

For `log1p_cites` and `log1p_total_patents`, the pre-period coefficients are often positive relative to `t = -1`, which means the pooled event studies should not be read as fully clean causal break designs. Part of the post-merger decline may reflect reversal from a stronger pre-treatment path rather than a sharp treatment-onset effect. The cited-patent outcomes show a similar issue, though in somewhat weaker form.

By contrast, outcomes such as `log1p_arriving_inventors_count` and `rolling_self_similarity` are somewhat more informative as mechanism variables. Their pre-periods look flatter, and the post-treatment weakening appears later and more gradually. Those are still not perfect designs, but they are easier to interpret as post-merger adjustment rather than simple continuation of a pre-existing trend.

### 3. How the baseline event studies and the sales-tercile triple-DiD fit together

Taken alone, the pooled baseline event studies are suggestive but not ideal headline evidence because several important outcomes show noticeable lead patterns. But once combined with the sales-tercile results, their role becomes clearer.

The pooled event studies tell us that there is **some average post-merger weakening**, especially in later years and especially for patent and citation outcomes. The triple-DiD event studies then explain why the pooled picture looks noisy: it is averaging together small acquirors that appear relatively vulnerable and large acquirors that appear substantially more resilient.

So the baseline event studies are still useful, but primarily as a descriptive first layer. The more publication-ready interpretation comes from reading them **together** with the heterogeneity results:
- pooled average effects are mixed and sometimes contaminated by pre-trend concerns,
- but the heterogeneity pattern is economically coherent,
- and it consistently points toward organizational capacity as the key moderator on the acquiror side.


## Why some results look mixed: economic and measurement intuition

The variation across outcomes and estimators is not just a statistical nuisance. Much of it is economically understandable.

### 1. Firm-level patenting is lumpy and sparse
Patenting is not smooth firm-year production. Even large public firms can have substantial year-to-year variation in patent counts, citation-weighted measures, and specialized patent buckets. That creates:
- many zeros in narrower outcomes,
- sensitivity to a small number of important projects,
- and greater estimator disagreement when comparisons rely on different sets of treated-control matches or different weighting schemes.

This is especially relevant for outcomes like `top10_to_2_patents`, `uncited_patents`, and `self_cites`, where small changes in a few patents can move the firm-year total considerably.

### 2. Citation outcomes arrive with lag
A patent filed in the post-treatment period does not reveal its citation profile immediately. That means citation-based outcomes can be contaminated by timing mismatch:
- some post-merger patents have not yet had enough time to accumulate citations,
- some observed citations reflect pre-merger projects that continue to mature,
- and event-time estimates can therefore blend project-timing effects with real treatment effects.

That timing issue helps explain why citation outcomes can differ more across methods than raw patent counts.

### 3. Mergers often change organization before they change measured output
The earliest merger effects may show up in:
- staffing,
- project selection,
- budget control,
- reporting lines,
- and portfolio rationalization,

before they show up in total patent output. That is one reason the heterogeneity results are so informative. They suggest that **who can absorb the organizational shock** matters at least as much as the average mechanical effect on patent counts.

### 4. Acquiror and target outcomes need not mirror each other
The acquiring firm is usually integrating new assets into a larger existing platform, while the target is being integrated into another organization. Those are very different adjustment problems. So it is entirely plausible that:
- acquiror results are moderated mostly by the acquiror’s scale and capacity,
- while target results are moderated more by how transformative the transaction is for the target.

### 5. Small-sample issues can matter even when the raw panel looks large
Some of the outcome-specific advanced-method samples are effectively much smaller than the full panel might suggest.
- SCM uses only treated units with acceptable pre-treatment fit, so successful fits can be a modest share of eligible treated firms.
- Dynamic estimators thin out at longer horizons as the number of usable treated cohorts shrinks.
- Narrower patent outcomes magnify this problem because the signal-to-noise ratio worsens.

So even with a reasonably large panel, some event-time or outcome-specific results can still be precision-limited.



## What the Sun–Abraham and CSDID files add

The additional SA and CSDID-related files are useful for evaluating credibility rather than just producing another coefficient.

### Sun–Abraham: some outcomes look cleaner than others
The acquiror SA files show an informative contrast.

- For **cited patents**, the pre-treatment coefficients at event times `-4`, `-3`, and `-2` are positive and non-trivial before the omitted period, while the post-treatment estimates drift toward zero or slightly negative values. That pattern suggests caution: part of the apparent post-merger decline may reflect a reversal from a stronger pre-treatment path rather than a clean break exactly at treatment.
- For **top10_to_2_patents**, pre-trends look more subdued, and the post-treatment estimates gradually become more negative, though modestly so.
- For **uncited patents**, pre-period coefficients are close to zero, while post-treatment coefficients become clearly negative from around event time `2` onward. This is the sort of pattern that is more naturally read as a dynamic post-treatment deterioration.
- For **self-cites**, pre-period coefficients are fairly flat, and the later post-treatment years turn negative. That is again suggestive of a delayed integration effect rather than an immediate jump at closing.

So SA does not only “confirm or reject” the baseline. It helps distinguish between:
- outcomes that may reflect underlying pre-treatment differences in trajectory,
- and outcomes where deterioration seems to emerge only after the merger.

### CSDID feasibility: identification support exists for the selected acquiror outcomes
The CSDID feasibility summaries are reassuring in one narrow but useful sense. For the selected acquiror outcomes, all cohort-time cells are identified in the feasibility files, with full cell coverage and non-trivial treated and control counts. That does not by itself prove the eventual causal estimate is clean, but it does show that the staggered-adoption design has adequate support for these outcomes rather than collapsing because of empty cohort-time cells.

That is worth mentioning in the notebook because it shows attention to design quality, not just point estimates.


## What the permutation placebo tests add

The permutation-placebo outputs are valuable because they ask a harder question than a standard pre-trend check: **if treatment timing is reassigned artificially, do we still recover “effects”?** In a setting with staggered adoption, lumpy patenting, and relatively short event windows, that is an important diagnostic. It helps separate findings that reflect treatment timing from findings that could also be generated by noisy dynamic structure in the panel.

The placebo evidence available so far is **cautionary rather than destructive**. It does not invalidate the project, but it does discipline what should be treated as headline material.

### 1. Baseline placebo evidence argues against overclaiming average effects

The uploaded placebo summary for the acquiror baseline DiD is limited, but that is already enough to make an important point. In this design, even a placebo timing assignment can generate a statistically significant coefficient for a firm-level innovation outcome. That means **baseline significance by itself is not sufficient** evidence in this setting. With many related outcomes, short windows, and noisy innovation measures, some equations will look significant under timing assignments that should have no causal meaning.

That does not mean the baseline framework is useless. It means the baseline should be treated as a **first-pass benchmark**, not as the final standard of proof.

### 2. Why this actually strengthens the final presentation

Once the placebo exercise is taken seriously, it becomes clearer why the notebook should emphasize:
- dynamic plots rather than isolated pooled coefficients,
- sign stability across multiple outcomes,
- heterogeneity patterns that align with economic theory,
- and agreement across estimators where available.

This is exactly why the current firm-level story should not be “we found a significant average effect of M&A on innovation.” The better story is more disciplined:

- average firm-level effects are often negative or mixed,
- some pooled event studies show non-trivial pre-trends,
- but the **heterogeneity pattern is persistent and economically coherent**,
- and that heterogeneity maps naturally into organizational capacity on the acquiror side and deal intensity on the target side.

So the placebo evidence lowers confidence in any single baseline coefficient, but it **raises the relative importance of the heterogeneity results**, which are now the strongest part of the firm-level discussion.

## Inventor-year baseline results: what they add beyond the firm panel

The inventor-year panel answers a different question from the firm-level analysis. The firm panel asks whether the **organization as a whole** changes its innovative output after a merger. The inventor-year panel asks whether **individual inventors linked to merger-exposed firms** change their own output, novelty, exploration behavior, or mobility relative to matched controls. That distinction is important for interpretation. Firm-level disruption can arise from composition effects, selective retention, project reallocation, or organizational mismatch even when not every incumbent inventor experiences the same change.

In that sense, the inventor-year results are best read as a **mechanism layer** for the broader firm-level findings.

### What the inventor-year baseline results suggest for acquirors

For the **acquiror inventor-year panel**, the baseline specifications with the most complete controls point to a fairly coherent first-pass pattern.

- In the pooled DiD, **exploration** is positive and statistically significant.  
- **Novelty** is also positive and statistically significant.  
- **`is_move_year`** is positive and statistically significant.  
- **`xi_real`** is strongly negative.  
- **Total patents** is not clearly different from zero in the baseline pooled specification.  

Taken literally, that combination says that inventors attached to acquirors appear to become **more exploratory and somewhat more mobile**, but not obviously more productive in a quality-adjusted sense. That is economically plausible. Mergers can widen the inventor's exposure to new technological combinations and new internal project assignments, while at the same time imposing integration frictions, match-specific disruption, and slower execution on the projects that matter most for measured quality.

The acquiror baseline **event studies** reinforce that interpretation, but they also make clear why caution is needed. For **exploration**, the post-treatment coefficients are mostly positive, which is consistent with the baseline DiD, but the lead coefficients are not perfectly flat. For **novelty**, the post-period is again mostly positive. For **`is_move_year`**, post-treatment coefficients are positive, but there are non-zero leads. For **`xi_real`**, the post-period becomes clearly and increasingly negative from roughly years 2 through 5. By contrast, **cites** and some other output-style measures show noticeable pre-trend movement, so they should not be treated as the cleanest inventor-level evidence on their own. fileciteturn6file0turn6file1turn6file2turn6file4 fileciteturn7file1turn7file3turn7file4turn7file6turn7file0

The best baseline reading for acquirors is therefore:

1. inventors appear to **search more broadly** after acquisition,  
2. inventor mobility may rise around the merger window,  
3. but quality-adjusted inventor performance, especially `xi_real`, appears to weaken.  

That is a meaningful mechanism result because it fits a story in which larger organizations reallocate inventors into new combinations and teams without necessarily preserving short-run efficiency or project continuity.

### What the inventor-year baseline results suggest for targets

For the **target inventor-year panel**, the baseline event studies point to a different and in some ways more contractionary pattern.

- **Exploration** is negative in the post-period for several event years.  
- **`is_move_year`** is generally lower after treatment in the event-study path.  
- **Total patents** is negative in later post years.  
- **`xi_real`** becomes negative mainly in later post years.  
- **Novelty** is weaker and more mixed than on the acquiror side.  

Economically, that difference between acquirors and targets is quite sensible. Inventors at target firms may face a more one-sided organizational shock: projects are absorbed into a larger structure, local autonomy may fall, and the existing research agenda may be interrupted rather than broadened. Acquiror inventors may be reallocated into a wider opportunity set; target inventors may be more likely to experience disruption without the same degree of internal optionality. fileciteturn7file9turn7file8turn7file10turn7file11

### Why these inventor-year baseline results are still valuable despite imperfect pre-trends

The most important econometric caveat is that the inventor-year baseline event studies are often **not flat in the pre-period**. In some outcomes the leads are large enough that they rule out an overly strong causal reading of the raw TWFE event-study path. That is especially true for acquiror `cites`, some movement-related paths, and several quality-composition outcomes. fileciteturn7file0turn7file3turn7file6

But those baseline results are still valuable for three reasons.

First, they help identify **where the action is**: movement, exploration, novelty, and quality-adjusted inventor performance look more informative than a simple “inventors patent more” story.

Second, the non-flat pre-period is itself informative. It suggests that inventors in merger-exposed firms are not randomly selected incumbents. They often come from firms or projects that were already evolving in economically meaningful ways before the transaction.

Third, the inventor-year baseline results help explain why the **firm-level** evidence looks heterogeneous and composition-sensitive. If inventors are already on different trajectories before the deal, and if post-deal adjustment works through reassignment, selective retention, and changing match quality, then it is not surprising that the most credible firm-level story is about heterogeneity rather than one universal average effect.


## Inventor-year advanced methods: what they add, and where they disagree

The advanced inventor-year estimators are essential because the baseline inventor event studies often have visible lead coefficients. The purpose of the advanced methods here is not simply to produce more numbers. It is to ask which inventor-level patterns survive estimators that treat staggered timing more carefully.

The key message from the uploaded inventor-year **BJS**, **CSDID**, and **Sun–Abraham** results is that the inventor-side evidence is **interesting but less settled than the firm-level heterogeneity results**. Some patterns recur across methods; others do not.

### 1. What looks most credible for acquiror inventors

For **acquirors**, the most defensible recurring pattern is a weakening in **`xi_real`**.

- The baseline pooled DiD is strongly negative.  
- The baseline event study turns increasingly negative in later post years.  
- The Sun–Abraham path is also negative in the post period, with the largest declines in later years.  
- The CSDID dynamic estimates are negative on average as well.  

This is one of the cleaner inventor-level results because the sign lines up across several methods, even though magnitudes differ. Economically, it suggests that merger-linked inventors at acquirors may be reallocated into broader search or transitional projects without preserving the same quality-adjusted performance as before. That interpretation fits a disruption-and-reallocation story better than a pure efficiency-gain story. fileciteturn6file4 fileciteturn7file6turn7file16turn7file10

For **exploration**, the evidence is more mixed but still informative.

- The baseline pooled DiD is positive.  
- The baseline event-study post path is mostly positive.  
- BJS is strongly positive in every uploaded post year.  
- Sun–Abraham is modestly positive in the post period.  
- CSDID, however, points in the opposite direction, with negative post-average effects.  

The most honest interpretation is not to force a single sign. Instead, the evidence suggests that acquiror-linked inventors do experience a meaningful change in their search behavior, but the precise sign and magnitude are **estimator-sensitive**. That can happen here because exploration is measured at the inventor-year level, matched support is thinner than in the firm panel, and the estimators weight cohorts and post periods differently. The baseline, BJS, and SA files lean toward **more exploration**, but the CSDID results are a serious reason to state that conclusion cautiously rather than as a hard headline. fileciteturn6file0 fileciteturn7file1turn7file12turn7file17turn7file13

For **novelty**, the picture is similar but slightly weaker.

- The baseline pooled DiD is positive.  
- The baseline event-study post path is mostly positive.  
- BJS is clearly positive in post years.  
- Sun–Abraham is also mildly positive.  
- CSDID again points negative on average.  

So novelty supports the same broad economic idea as exploration—post-merger inventor behavior may shift toward broader or less routine combinations—but the estimator disagreement means it should be presented as **suggestive mechanism evidence**, not as the single strongest inventor conclusion. fileciteturn6file2 fileciteturn7file4turn7file14turn7file18turn7file13

For **mobility (`is_move_year`)**, the advanced methods again disagree.

- The baseline pooled and baseline event-study results suggest a positive acquiror-side mobility response.  
- BJS is clearly positive in every post year.  
- Sun–Abraham is positive early in the post period and then fades.  
- CSDID points negative on average.  

That is not enough to headline a robust mobility effect for acquiror inventors. It is enough to say that **mobility is clearly one of the dimensions along which merger adjustment occurs**, but that the exact sign is sensitive to estimator and weighting scheme. fileciteturn6file1 fileciteturn7file3turn7file13turn7file18turn7file14

### 2. What looks most credible for target inventors

The target-inventor evidence is also mixed, but the baseline and Sun–Abraham results are more often consistent with a **weaker post-merger inventor environment** rather than a broadened one.

For **total patents**, the baseline target event study becomes negative in later post years, and Sun–Abraham also turns more negative in later years. CSDID, by contrast, is strongly positive on average. That disagreement makes it hard to state a robust inventor-level output conclusion for targets. The safest reading is that target inventor output is **not stably positive across estimators**, and the baseline plus SA evidence leaves open the possibility of later deterioration. fileciteturn7file10turn7file21turn7file19

For **`xi_real`**, the baseline target event study becomes negative in later post years, while Sun–Abraham is mixed and CSDID is positive on average. Again, this is not settled enough to claim a single robust target-inventor quality effect. But it does suggest that any target-side gains are far from universal. fileciteturn7file11turn7file22turn7file20

For **exploration**, the baseline target event study is mostly negative after treatment, while CSDID is essentially centered near zero on average with substantial variation by event time. That makes the most defensible statement a modest one: the target inventor data do not support a strong, robust increase in exploratory search after the deal. fileciteturn7file9turn7file15

For **`is_move_year`**, the target results are again not cleanly aligned. The baseline path is mostly negative post-treatment, BJS is positive, SA is mixed, and CSDID is positive on average. That is too inconsistent to state a confident target-side mobility story from the current files alone. fileciteturn7file8turn7file13turn7file20turn7file18

### 3. How to interpret the disagreement across advanced estimators

This disagreement should be acknowledged explicitly. I do not think it invalidates the inventor-year part of the project, but it does discipline what should be treated as a headline.

There are several reasons this disagreement is not surprising:

1. **The inventor-year sample is much more granular and support is thinner.**  
   Different estimators can end up using meaningfully different treated-control comparisons once cohort support, matching restrictions, and post-treatment horizons are imposed.

2. **The estimators weight dynamics differently.**  
   BJS is an imputation estimator focused on untreated potential outcomes; Sun–Abraham aggregates cohort-specific relative-time effects; CSDID can be especially sensitive to support and cohort composition in sparse cells. The same underlying data can therefore generate different average summaries even when several estimators agree on part of the dynamic path.

3. **The inventor outcomes are behaviorally complex.**  
   Exploration, novelty, and move probability are not smooth production measures. They are affected by reassignment, project mix, internal hierarchy, and retention dynamics. It is therefore plausible that different estimators pick up different slices of the same underlying organizational adjustment.

### Bottom line from the advanced inventor methods

The advanced inventor results support a careful, job-market-ready conclusion:

- the inventor panel contains **real mechanism information**,  
- some inventor outcomes—especially **`xi_real` for acquirors**—look meaningfully affected,  
- but several inventor claims are **estimator-sensitive**, so the inventor panel should be presented as a mechanism layer that complements the stronger firm-level heterogeneity results, not as the sole centerpiece of the project.

That is not a weakness if stated clearly. In a recruiting context, it shows that the notebook does not hide estimator disagreement. It uses that disagreement to sort the inventor findings into **more robust**, **suggestive**, and **still uncertain** categories.


## Additional discussion: inventor-year event studies (baseline and advanced estimators together)

The inventor-year event-study evidence is most useful when read as a **set of dynamic diagnostics**, not as a single preferred estimator. The baseline TWFE event studies tell us where the interesting action is. BJS, CSDID, and Sun–Abraham then tell us which of those dynamic patterns look stable enough to emphasize.

### Acquiror inventors: the most coherent dynamic story is “broader search, weaker quality-adjusted performance”

For **acquiror inventors**, the most intuitive synthesis is that inventors appear to move toward somewhat broader search and different project combinations after the deal, but that quality-adjusted performance weakens.

The best evidence for the “broader search” part is that:

- baseline and SA event studies for **exploration** are mostly positive post-treatment,  
- baseline and SA event studies for **novelty** are also mostly positive post-treatment,  
- and BJS delivers clearly positive post-treatment effects for both outcomes.  

That combination is hard to dismiss entirely, even though CSDID points in the opposite direction. The safest wording is therefore that the post-merger inventor adjustment at acquirors is **consistent with increased recombination or exploratory search**, but that the exact magnitude is not fully robust across estimators. fileciteturn7file1turn7file4turn7file12turn7file14turn7file17turn7file18

The best evidence for the “weaker quality-adjusted performance” part is the much more consistent **`xi_real`** pattern. The baseline event study, the SA event study, and the CSDID post-average all point negative, and the baseline pooled DiD is strongly negative as well. That is the inventor-side result I would trust most. fileciteturn6file4 fileciteturn7file6turn7file16turn7file10

For **cites**, the dynamic path is more fragile. The baseline event study becomes strongly positive in later post years, but it also has non-flat leads. SA is mixed, and CSDID is positive on average. That is enough to treat citations as suggestive but not as the cleanest inventor outcome. fileciteturn7file0turn7file12turn7file17

For **`is_move_year`**, the dynamic evidence says mobility is part of the adjustment process, but it does not deliver one clean sign across all estimators. That again pushes the notebook toward a careful mechanism interpretation rather than a simple mobility headline. fileciteturn7file3turn7file13turn7file18

### Target inventors: the dynamic evidence is weaker and less stable

For **target inventors**, the baseline event studies often suggest weaker post-treatment inventor outcomes, especially for **exploration**, **total patents**, and later **`xi_real`**. The SA target results broadly lean the same way for total patents and are at least consistent with a fragile or deteriorating post-treatment target path. But CSDID frequently points the other way for total patents and `xi_real`, while BJS is positive for some uploaded target outcomes. fileciteturn7file9turn7file10turn7file11turn7file19turn7file20turn7file21turn7file22

That means the target inventor panel is **not yet a strong standalone headline**. What it does add is directional support for the broader idea that target inventors face a more contractionary and unstable post-merger environment than acquiror inventors. But the evidence is not stable enough across estimators to make a very sharp target-side inventor claim.

### What this means for the notebook narrative

The inventor-year event studies therefore do not overturn the core narrative established in the firm panel. They refine it.

The most convincing overall reading is:

1. **firm-level heterogeneity remains the strongest headline result**,  
2. the inventor-year panel provides economically meaningful mechanism evidence,  
3. and within that mechanism evidence, the most credible inventor-side result is that **acquiror inventors appear to shift toward broader search while suffering weaker quality-adjusted performance**.

That is a nuanced conclusion, but it is also exactly the sort of conclusion that plays well in a serious applied-micro or tech-economist setting. It does not oversell noisy event-study paths, yet it extracts a clear organizational message: mergers can redirect inventive labor into different combinations and projects without guaranteeing that the resulting inventor-level performance is equally effective on all margins.

### How I would present this in a recruiting or job-market context

I would not present the inventor-year panel as “the one definitive causal estimate.” I would present it as the **mechanism complement** to the stronger firm-level heterogeneity results:

> “At the firm level, the clearest result is that post-merger innovation effects are heterogeneous and depend on organizational capacity and deal intensity. At the inventor level, the evidence suggests that mergers reallocate inventive labor and change search behavior, especially on the acquiror side, but the inventor responses are estimator-sensitive. The strongest inventor-side result is weaker quality-adjusted performance, alongside suggestive evidence of broader search and changing mobility.”

That framing is both honest and strong. It shows that the project is not only about estimating average treatment effects. It is about using multiple panels and estimators to understand **how organizational shocks reshape innovative activity at both the firm and inventor level**.


## Additional discussion: inventor-year triple-DiD by inventor standing within the firm

The newly uploaded inventor-year triple-DiD files add a second and, in some ways, even more interesting heterogeneity dimension. Instead of splitting inventors by the size of the acquiring firm, these specifications split them by whether the inventor is in the **upper half of the within-firm cumulative patent distribution**. Economically, this is a proxy for an inventor's standing inside the firm's innovation hierarchy: roughly, more established and central inventors versus less established inventors inside the same organization.

That is an important complement to the sales-based heterogeneity analysis. The sales split asks whether **firm absorptive capacity** matters. The within-firm patent-rank split asks whether **inventor position inside the organization** matters. Together, they move the project beyond the question of whether M&A changes innovation on average and toward the richer question of **who inside the firm adjusts, and under what organizational conditions**.

### Why this split is economically meaningful

Inventors who are already high in the internal patent distribution are likely to differ from lower-ranked inventors along several dimensions:

- they may control more mature projects or established technology trajectories,
- they may have more internal bargaining power and more stable team attachments,
- they may be more embedded in the pre-merger organization and therefore more exposed to integration frictions,
- and they may have less upside from being reallocated into new exploratory work than junior or peripheral inventors.

That makes this heterogeneity design especially useful for interpreting the mechanism. If post-merger changes are concentrated among lower-ranked inventors, the story is less about a universal productivity shock and more about **organizational reallocation**. If they are concentrated among higher-ranked inventors, the story looks more like disruption of the firm's core inventive capital.

### What the new results suggest overall

Taken as a group, these files reinforce the main conclusion already emerging in the notebook: the inventor-year panel does **not** support a clean, homogeneous treatment effect. Instead, it suggests that post-merger adjustment differs sharply across types of inventors.

The most consistent pattern in these within-firm-rank splits is the following:

1. the omitted group often exhibits **strong pre-period movements**, especially in quantity-based outcomes,  
2. the interaction terms imply that inventors in the upper half of the within-firm patent distribution often move **less extremely** than the omitted group, and  
3. the post-merger response is often concentrated in the lower-ranked group or reflects a relative reallocation away from them and toward more established inventors, depending on the outcome.

That makes economic sense. Large organizational changes rarely affect all inventors symmetrically. They tend to change who receives resources, which projects are preserved, and which inventors are retained, reassigned, or sidelined.

### Quantity outcomes: strong reallocation, not a uniform inventor-level productivity effect

The strongest evidence in these files comes from the quantity measures.

For **total patents**, the omitted group displays very large positive post-period coefficients after the merger, but also very large negative pre-period coefficients, while the interaction for inventors in the upper half of the within-firm patent distribution is strongly positive before the merger and strongly negative after it. Interpreted in levels, this means the omitted group is driving the large post-merger increase, whereas more established inventors experience a much smaller post-merger rise and in some periods a materially weaker path overall. This is a clear sign of heterogeneity, but it is also a sign that the average path is being shaped by strong composition and trend differences rather than a single common treatment effect.

For **citations**, the same qualitative pattern appears. The omitted group shows large positive post-merger effects, while the upper-half interaction turns strongly negative after treatment. Again, the natural interpretation is not that mergers make all inventors uniformly more cited. Rather, citation dynamics appear to shift toward a subset of inventors, while more established inventors exhibit a weaker post-merger response than the omitted group.

From a substantive perspective, this is plausible. Junior or lower-ranked inventors may be reassigned into active post-merger integration projects, may benefit from broader internal collaboration, or may simply account for more of the measured transition in patenting because they are more marginal and more elastic. By contrast, highly established inventors may already sit on mature trajectories that are less expandable after the deal, or they may be disrupted by changes in priorities.

### Novelty and search direction: core inventors look more stable

The **novelty** and **exploration/exploitation** files are especially useful because they help distinguish pure quantity changes from changes in the type of innovative search.

For **exploration**, the omitted group becomes increasingly more exploratory after treatment, but the interaction for upper-half inventors is strongly negative in the post-period. This suggests that the rise in exploration is disproportionately concentrated among lower-ranked inventors, while more established inventors shift much less in that direction. That is a sensible organizational story: new or peripheral inventors may be easier to redeploy into boundary-crossing or exploratory projects, while central inventors remain attached to incumbent lines of research.

For **exploitation**, the mirror image appears. The omitted group becomes more exploitative over time, but upper-half inventors show an offsetting negative interaction, especially in later post years. This does not mean the more established inventors become exploratory in absolute terms; rather, it means their post-merger movement toward exploitative production is weaker than in the omitted group. Read together with the exploration result, the interpretation is that lower-ranked inventors seem to bear more of the post-merger behavioral adjustment, while upper-ranked inventors look comparatively insulated or stabilized.

For **novelty**, the interaction terms are weaker than for patents or citations, but the later positive interaction coefficients suggest that upper-half inventors may preserve novelty somewhat better in later post years. That is an economically reasonable result: once integration turbulence settles, the most established inventors may be better positioned to convert organizational stability and access to resources into higher-quality or more novel inventions.

### Mobility: merger adjustment appears more pronounced for lower-ranked inventors

The **is_move_year** specification is also informative. The omitted group shows a pattern of declining mobility in the lead period followed by positive post-merger movement, while the interaction for upper-half inventors offsets some of that post-period increase and becomes increasingly positive only later. The broad message is that inventor mobility is not moving identically across the internal hierarchy.

That, too, is plausible. Lower-ranked inventors are generally more weakly attached to the organization and may be more vulnerable to post-merger disruption or more willing to move when the internal opportunity set changes. Higher-ranked inventors may initially be retained more aggressively, either because they are more valuable or because they sit on strategically important projects. Later mobility among high-ranked inventors may then emerge only once the long-run organizational consequences of the merger become clear.

### What this adds to the credibility of the overall project

These within-firm-rank results do not rescue the simple average inventor-year event study; they do something more useful. They show that the mixed average evidence is not just noise. It is at least partly the consequence of **meaningful internal heterogeneity**.

That is valuable for a recruiting audience because it demonstrates three things:

1. **The empirical work is not confined to one mechanical specification.**  
   The project uses the data structure to ask a genuinely richer organizational question.

2. **The economic interpretation follows the heterogeneity, not just significance stars.**  
   The results are being read through the lens of organizational economics, inventor hierarchy, and absorptive capacity.

3. **The notebook is honest about where identification is stronger and weaker.**  
   The within-firm-rank split reveals a real pattern, but it also highlights that the inventor-year panel is heavily affected by non-flat leads and reallocation dynamics. That is exactly the kind of disciplined interpretation one would want from a PhD-level empirical researcher.

### Best way to present these results publicly

For the notebook and any LinkedIn-style summary, the cleanest framing is:

- the **firm panel** shows that merger-related innovation effects are heterogeneous across firms,
- the **inventor panel** shows that the adjustment also differs across inventors within firms,
- and the most credible inventor-level result is therefore not a universal treatment effect but a pattern of **organizationally structured reallocation**.

In other words, the project increasingly looks like evidence that M&A changes the internal allocation of inventive activity, and that the direction and magnitude of that change depend both on firm characteristics and on the inventor's place in the internal innovation hierarchy.

That is a stronger and more sophisticated story than simply claiming that mergers raise or lower innovation on average.



## Why the heterogeneity story remains the most credible result

The inconsistency in some average treatment effects does **not** make the project weak. If anything, it makes the stronger heterogeneity evidence more valuable.

Why?

1. **Average effects are clearly not uniform.**  
   That is already an important empirical finding in a setting where theory would not predict uniform adjustment across all firms.

2. **The moderation pattern is systematic rather than ad hoc.**  
   Acquiror results repeatedly emphasize firm size and capacity; target results repeatedly emphasize relative deal scale.

3. **The machine-learning results align with the parametric design.**  
   In the causal-forest summaries uploaded so far, `lag1_log_sale` is repeatedly the most important or one of the most important heterogeneity drivers for acquiror outcomes, and it also ranks highly for target outcomes. That is exactly what one would hope to see if the parametric triple-DiD is capturing a real moderator rather than data mining noise.

4. **The mechanism interpretation is compelling.**  
   Mergers do not only combine assets; they combine decision rights, reporting structures, project portfolios, and inventor teams. A story centered on absorptive capacity and transaction intensity is therefore more plausible than a one-size-fits-all average effect.

That is why the recruiting-oriented presentation should lean into the idea that this project studies **which firms absorb innovation shocks better and why**, not merely whether one average coefficient is above or below zero.


## Recommended main story for the notebook and GitHub project

I would recommend the following narrative.

### Main claim
**M&A does not appear to generate one uniform innovation effect. Instead, the evidence points to post-merger disruption whose consequences depend strongly on organizational capacity, portfolio composition, and deal scale.**

### Supporting interpretation
- On average, pooled baseline DiD and event-study estimates often show weaker post-deal innovation outcomes, but those average effects are uneven across outcomes and sometimes complicated by pre-trend concerns.
- For **acquirors**, the strongest and most robust pattern is that **larger firms are better able to absorb disruption**. Baseline sales is the clearest acquiror-side moderator across patents, citations, selected inventor-flow outcomes, and `xi_real`.
- For **targets**, the strongest robust pattern is that **larger deals relative to the target create stronger disruption**.
- On the acquiror side, the additional patent-composition results suggest that post-merger weakening is not only about raw counts. Outcomes such as `cited_patents`, `uncited_patents`, and `rolling_self_similarity` indicate changes in the composition and continuity of innovative activity.
- Mobility and self-similarity outcomes suggest reorganization, but they should be framed as mechanism evidence rather than as the sole headline claim.
- The placebo results imply that isolated baseline significance should not be overemphasized, which makes the heterogeneity results even more central to the final presentation.

### Why this is a strong recruiting story
This is the kind of result a tech economist or applied microeconomist should want to communicate:
- it starts with a broad descriptive benchmark,
- it checks robustness with multiple estimators,
- it distinguishes average effects from heterogeneity,
- it treats noisy outcomes with appropriate caution,
- and it turns a complex empirical setting into a disciplined statement about absorptive capacity, integration, and innovation organization.

## Draft text for the notebook’s expanded main results section

**Expanded draft results discussion**

The firm-level results suggest that mergers are associated with meaningful disruption to innovative activity, but the disruption is not uniform across firms or outcomes. In the pooled stacked DiD and baseline event studies, both acquirors and targets frequently exhibit weaker post-treatment innovation outcomes, including patents, citation-based measures, and selected inventor-flow variables. That first-pass evidence is economically plausible: mergers can interrupt research pipelines, reassign teams, rationalize overlapping projects, and divert managerial attention toward integration tasks rather than innovative production.

The more important result, however, is not the pooled average decline itself. Once the analysis moves to heterogeneity designs and more specialized estimators, the cleaner message is that the effect of M&A depends strongly on **who is absorbing the shock** and **how large the transaction is relative to the affected organization**.

On the **acquiror** side, baseline firm size emerges as the most consistent moderator. The sales-based triple-DiD results are now supported by both the pooled coefficient tables and the dynamic event-study interactions. Smaller acquirors appear more exposed to post-merger disruption, while larger acquirors are better able to preserve innovation performance. That does not mean large acquirors experience a post-merger innovation boom. A more careful reading is that organizational scale, managerial depth, and absorptive capacity make them **more resilient** when integration begins to disrupt projects and teams.

On the **target** side, the more convincing moderator is deal size relative to target scale. The larger the transaction relative to the target, the more severe the apparent disruption. That asymmetry between acquirors and targets is one of the most interesting economic conclusions in the notebook. It suggests that merger integration is governed by different bottlenecks on the two sides of the transaction: acquirors are constrained mainly by their ability to absorb and govern the combined organization, while targets are disrupted mainly by how transformative the transaction is relative to their own scale.

The newer acquiror files also add useful nuance on **which firm-level outcomes are most affected**. The baseline event studies do not show a single sharp collapse at closing. Instead, several key outcomes, including `log1p_cites`, `log1p_total_patents`, `log1p_uncited_patents`, and `rolling_self_similarity`, weaken gradually in the post-treatment years. That pattern is economically sensible. Research pipelines often continue to produce in the short run, while the organizational effects of integration accumulate more slowly through project reprioritization, team reshuffling, and changes in technological continuity.

At the same time, the dynamic evidence also shows why caution is needed. Several pooled event studies, especially for patent and citation outcomes, display noticeable positive lead coefficients relative to the omitted `t = -1` period. That means the average event-study plots should not be treated as perfectly clean causal break designs. They are still useful descriptive evidence, but they are strongest when interpreted together with the heterogeneity results, which explain why the pooled paths are noisy.

The acquiror-side `Z_deal_rel_q3` results strengthen this interpretation by contrast. Relative deal value is simply much weaker as a moderator for acquirors than baseline size. The few suggestive interactions are concentrated in organizational or mechanism-style outcomes rather than the core patent and citation measures. This makes the final story more disciplined, not less: for acquirors, the decisive margin is not whether the deal is large relative to market value in a mechanical sense, but whether the acquiror has the organizational capacity to absorb the shock.

The placebo exercises reinforce the same lesson. In this setting, a single significant baseline coefficient should not be treated as decisive. Once placebo timing can also generate significance in some equations, the right standard becomes broader: do the signs line up across related outcomes, do the dynamic patterns make economic sense, and do the heterogeneity results map onto a plausible mechanism? On those criteria, the firm-level results are now much stronger than a narrow focus on any one pooled estimate would suggest.

The inventor-year panel adds a useful mechanism layer. It suggests that merger adjustment works not only through aggregate firm reorganization, but also through changes in inventor behavior, search direction, and mobility. The most credible inventor-side pattern is on the **acquiror** side: inventors appear to shift toward broader search and somewhat higher novelty, while quality-adjusted inventor performance—especially `xi_real`—weakens after the deal. That interpretation is supported by the baseline inventor results and by several of the advanced inventor estimators, though not all of them point in exactly the same direction. For targets, the inventor-level evidence is weaker and more estimator-sensitive, so it is better treated as suggestive supporting evidence than as a standalone headline.

That final point is worth stating explicitly. The inventor panel is informative, but it is not as clean as the best firm-level heterogeneity results. Several inventor event studies display non-flat lead coefficients, and the advanced estimators sometimes disagree in sign or magnitude. I do not think that undercuts the project. Instead, it sharpens the correct conclusion: the strongest evidence in the notebook is that **M&A affects innovation through heterogeneous organizational adjustment**, and the inventor-year results help illustrate the mechanism by showing how inventive labor itself appears to be reallocated and disrupted.

Overall, the publication-ready interpretation is therefore not that mergers have one universal innovation effect. It is that **mergers create disruption whose consequences depend systematically on organizational capacity, deal intensity, and the composition of innovative activity**. Larger acquirors are better able to absorb the disruption. Larger relative deals are more disruptive for targets. And the inventor-year evidence suggests that at least part of this organizational adjustment operates through changes in inventor search behavior, mobility, and match quality rather than only through a simple firm-wide change in patent counts.


In [ ]:
# Optional convenience cell:
# once the heavy analysis has been run and outputs exist, use the helper file
# to create notebook-ready support CSVs in one step.
#
# support = write_notebook_support_outputs(
#     output_root=MODEL_OUT,
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# support